# SGJobData — Step-by-Step Data Cleaning

Loads `data/raw/SGJobData.csv` and cleans it one documented step at a time.
Every step prints a **BEFORE / AFTER** comparison so the effect of the change is visible,
and records its numbers into the `M` metrics dict.

The final cell writes `docs/data-cleaning-report-generated.md` from those live numbers, with
sections for **type conversions**, **ghost rows**, **data fixing**, **data filling**,
**duplicate rows**, **feature enrichment** and **column pruning**.

Nothing here is hard-coded from a previous run: re-running on a refreshed extract produces a
report that matches the new data.

**Pipeline order** — junk rows first (so column statistics are computed on real postings),
then structure, then values, then dtypes:

```
load → ghost rows → synthetic rows → prune dead columns → dates → categories JSON
     → salary fixes → text normalisation → filling decisions → duplicates
     → categorical + downcast → validate + save → feature enrichment
     → prune low-value derived columns → report
```

Note the two prunes sit at opposite ends, and that is forced rather than stylistic. Step 4 removes
columns the *source* shipped dead, early, so every later statistic is computed on real columns.
Step 14 removes columns the *pipeline itself* creates, and it cannot run any earlier because those
columns do not exist until Step 13 builds them — `competitiveness_score` is derived from
`skill_count`, which is derived from `title`. A column has to exist before it can be measured and
rejected.

Outputs `jobs_cleaned.parquet` (source column names), `job_category.parquet` (the many-to-many
bridge) and `jobs_enriched.parquet` (renamed and derived to match `src/pipeline/` and `JOBS_SCHEMA`).

## 0 · Setup

`SALARY_FLOOR` and `SALARY_CEILING` are the two judgement calls in the whole pipeline.
They are declared here, in one place, so the report can state exactly what was assumed.

In [19]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd

RAW_CSV     = Path('../data/raw/SGJobData.csv')
OUT_DIR     = Path('../data/processed')
REPORT_PATH = Path('../docs/data-cleaning-report-generated.md')
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- tunable cleaning parameters (surfaced in the report) ---------------------
SALARY_FLOOR         = 500      # monthly SGD; at or below this a salary is a placeholder, not a wage
SALARY_CEILING       = 100_000  # monthly SGD; above this it is a data-entry error
INTERN_STIPEND_FLOOR = 300      # monthly SGD; below SALARY_FLOOR but a plausible internship stipend
SYNTHETIC_ID_RE      = r'^RANDOM_JOB_'
EXPERIENCE_MAX       = 100       # years; above this the value is not physically possible

# --- JOBS_SCHEMA columns that are dead in THIS extract (see Step 13) --------------------------
DEAD_COLUMNS = {
    'location':        'the extract is Singapore-only - zero variance',
    'salary_currency': "restates salary_type, itself constant 'Monthly' here and dropped in Step 4",
    'description':     'the source CSV has no description field at all',
    'requirements':    'the source CSV has no requirements field at all',
    'created_at':      'records when the pipeline ran, not a property of the posting; '
                       'DatabaseManager.insert_jobs stamps it at insert time',
}

pd.set_option('display.width', 200, 'display.max_columns', 60, 'display.max_rows', 80)

metrics = {'steps': [], 'params': {'SALARY_FLOOR': SALARY_FLOOR, 'SALARY_CEILING': SALARY_CEILING,
                             'INTERN_STIPEND_FLOOR': INTERN_STIPEND_FLOOR,
                             'SYNTHETIC_ID_RE': SYNTHETIC_ID_RE, 'EXPERIENCE_MAX': EXPERIENCE_MAX},
     'dead_schema_cols': dict(DEAD_COLUMNS)}
print('pandas', pd.__version__, '| numpy', np.__version__)
print('dead columns (never written to either output):')
for _c, _why in DEAD_COLUMNS.items():
    print(f'   {_c:<16} {_why}')

pandas 3.0.5 | numpy 2.4.6
dead columns (never written to either output):
   location         the extract is Singapore-only - zero variance
   salary_currency  restates salary_type, itself constant 'Monthly' here and dropped in Step 4
   description      the source CSV has no description field at all
   requirements     the source CSV has no requirements field at all
   created_at       records when the pipeline ran, not a property of the posting; DatabaseManager.insert_jobs stamps it at insert time


### Comparison helpers

`snap()` takes a cheap fingerprint of a frame (rows, columns, memory, NaN cells) instead of a
full `.copy()`, so a multi-hundred-MB frame can be compared at every step without doubling memory.
`compare()` prints the before/after block and files the delta into `M['steps']` for the report.

In [20]:
def snapshot(frame):
    'Cheap fingerprint of a DataFrame - avoids copying 400 MB at every step.'
    return {'rows': len(frame), 'cols': frame.shape[1],
            'mem': round(frame.memory_usage(deep=True).sum() / 1e6, 1),
            'na': int(frame.isna().sum().sum()), 'columns': list(frame.columns)}


def compare(before, after, title, note=''):
    'Print a BEFORE/AFTER block for one cleaning step and record it in metrics.'
    after_stats = after if isinstance(after, dict) else snapshot(after)
    print(f'=== {title} ===')
    if note:
        print(f'    {note}')
    hdr = f'{"":<7} {"rows":>11} {"cols":>6} {"mem MB":>9} {"NaN cells":>12}'
    print(hdr)
    print(f'{"BEFORE":<7} {before["rows"]:>11,} {before["cols"]:>6} {before["mem"]:>9,.1f} {before["na"]:>12,}')
    print(f'{"AFTER":<7} {after_stats["rows"]:>11,} {after_stats["cols"]:>6} {after_stats["mem"]:>9,.1f} {after_stats["na"]:>12,}')
    print(f'{"DELTA":<7} {after_stats["rows"]-before["rows"]:>+11,} {after_stats["cols"]-before["cols"]:>+6} '
          f'{after_stats["mem"]-before["mem"]:>+9,.1f} {after_stats["na"]-before["na"]:>+12,}')
    removed = [c for c in before['columns'] if c not in after_stats['columns']]
    added   = [c for c in after_stats['columns'] if c not in before['columns']]
    if removed: print('    columns removed:', removed)
    if added:   print('    columns added  :', added)
    print()
    metrics['steps'].append({'step': title, 'note': note,
                       'rows_before': before['rows'], 'rows_after': after_stats['rows'],
                       'cols_before': before['cols'], 'cols_after': after_stats['cols'],
                       'mem_before': before['mem'], 'mem_after': after_stats['mem'],
                       'removed': removed, 'added': added})
    return after_stats


def to_markdown(frame, index=True):
    'DataFrame -> markdown table (avoids a tabulate dependency).'
    frame = frame.reset_index() if index else frame
    cols = [str(c) for c in frame.columns]

    def fmt(v):
        if isinstance(v, (bool, np.bool_)):        return str(bool(v))
        if isinstance(v, float) and not np.isnan(v):
            return f'{v:,.2f}'.rstrip('0').rstrip('.') if abs(v) < 1e15 else f'{v:,.0f}'
        if isinstance(v, (int, np.integer)):       return f'{v:,}'
        return str(v)

    body = '\n'.join('| ' + ' | '.join(fmt(v) for v in rec) + ' |' for rec in frame.itertuples(index=False))
    return ('| ' + ' | '.join(cols) + ' |\n'
            '| ' + ' | '.join('---' for _ in cols) + ' |\n' + body)


FLOAT32_HALF_LIMIT = 2 ** 23  # above this, float32 cannot represent a .5 fraction


def float32_check(series):
    'Round-trip error of a float32 cast, and how many .5 values sit above what it can represent.'
    values = series.astype('float64')
    # notna() first: NaN % 1 != 0 evaluates True, which would count every nulled salary as
    # carrying a fraction.
    fractional = values.notna() & (values % 1 != 0)
    error = (values.astype('float32').astype('float64') - values).abs()
    return {
        'max_error': float(error.max()),
        'worst_value': float(values[error.idxmax()]) if error.max() > 0 else float('nan'),
        'max_value': float(values.max()),
        'half_rows': int(fractional.sum()),
        'half_unrepresentable': int((fractional & (values > FLOAT32_HALF_LIMIT)).sum()),
    }


def column_profile(frame):
    'Per-column dtype / missingness / zero-inflation profile.'
    numeric_cols = frame.select_dtypes(include='number').columns
    return pd.DataFrame({
        'dtype':    frame.dtypes.astype(str),
        'na_count': frame.isna().sum(),
        'na_pct':   (frame.isna().mean() * 100).round(2),
        'n_unique': frame.nunique(dropna=True),
        'zeros':    [int((frame[c] == 0).sum()) if c in numeric_cols else 0 for c in frame.columns],
    })

## 1 · Load the raw CSV

`read_csv` with no `dtype=` map, deliberately — the point of this notebook is to show what the
untyped load costs and to fix it explicitly rather than hide it in a parser argument.

In [21]:
df = pd.read_csv(RAW_CSV)
raw_stats = snapshot(df)
metrics['raw'] = raw_stats

print(f'loaded {raw_stats["rows"]:,} rows x {raw_stats["cols"]} cols, {raw_stats["mem"]:,.1f} MB deep')

# baseline for the float32 question revisited in Step 11 - measured on untouched raw values
metrics['float32_raw'] = float32_check(df['average_salary'])
print(f'float32 round-trip error on raw average_salary: {metrics["float32_raw"]["max_error"]} '
      f'({metrics["float32_raw"]["half_unrepresentable"]} .5 value(s) above '
      f'${FLOAT32_HALF_LIMIT:,}, which float32 cannot represent)')

# displayed for the reader; not stored - the report renders the FINAL profile, not this one
column_profile(df).sort_values('na_count', ascending=False)

loaded 1,048,585 rows x 22 cols, 401.7 MB deep
float32 round-trip error on raw average_salary: 0.5 (1 .5 value(s) above $8,388,608, which float32 cannot represent)


,dtype,na_count,na_pct,n_unique,zeros
occupationId,float64,1048585,100.00,0,0
categories,str,3988,0.38,21125,0
metadata_expiryDate,str,3988,0.38,453,0
title,str,3988,0.38,377084,0
metadata_jobPostId,str,3988,0.38,1044597,0
metadata_newPostingDate,str,3988,0.38,431,0
metadata_originalPostingDate,str,3988,0.38,603,0
status_jobStatus,str,3988,0.38,3,0
salary_type,str,3988,0.38,1,0
employmentTypes,str,3988,0.38,8,0


## 2 · Ghost rows

A *ghost row* is a structurally empty record: every text field `NaN` **and** every numeric field
exactly `0`. The evidence for treating them as junk rather than as partially-missing postings is
that the per-row NaN count is **bimodal** — a row is either fully populated or fully blank, never
in between. If these were real postings with patchy collection, we would see intermediate counts.

They are removed first so that every column statistic computed later describes real postings.

> Note the trap this avoids: at least one column is entirely NaN (`occupationId`, in this extract),
> so `df.dropna()` on the raw frame returns an **empty** DataFrame. An explicit mask states the intent and cannot misfire.

In [22]:
num_cols  = list(df.select_dtypes(include='number').columns)
bool_cols = list(df.select_dtypes(include='bool').columns)
text_cols = [c for c in df.columns if c not in num_cols + bool_cols]

na_per_row = df[text_cols].isna().sum(axis=1)
print('NaN-count-per-row distribution across the', len(text_cols), 'text columns:')
print(na_per_row.value_counts().sort_index().to_string(), '\n')

ghost = (na_per_row == len(text_cols)) & (df[num_cols].fillna(0) == 0).all(axis=1)

metrics['ghost'] = {
    'n': int(ghost.sum()),
    'pct': round(ghost.mean() * 100, 3),
    'text_cols': len(text_cols),
    'bimodal': sorted(int(v) for v in na_per_row.unique()),
    'numeric_abs_sum': float(df.loc[ghost, num_cols].abs().sum().sum()),
    'idx_min': int(df.index[ghost].min()), 'idx_max': int(df.index[ghost].max()),
    'partial': int(((na_per_row > 0) & (na_per_row < len(text_cols))).sum()),
    'vacancies_zero_elsewhere': int(((df['numberOfVacancies'] == 0) & ~ghost).sum()),
}
print('ghost rows                      :', f'{metrics["ghost"]["n"]:,}', f'({metrics["ghost"]["pct"]}%)')
print('rows with a PARTIAL NaN pattern :', metrics['ghost']['partial'], '<- 0 proves the pattern is all-or-nothing')
print('sum |numeric| over ghost rows   :', metrics['ghost']['numeric_abs_sum'], '<- every numeric field is exactly 0')
print('rows with 0 vacancies elsewhere :', metrics['ghost']['vacancies_zero_elsewhere'])
print('index span                      :', metrics['ghost']['idx_min'], '-', metrics['ghost']['idx_max'])

before = snapshot(df)
df = df.loc[~ghost].copy()
compare(before, df, 'Step 2 - remove ghost rows',
        f'{metrics["ghost"]["n"]:,} structurally empty records dropped')

NaN-count-per-row distribution across the 11 text columns:
0     1044597
11       3988 

ghost rows                      : 3,988 (0.38%)
rows with a PARTIAL NaN pattern : 0 <- 0 proves the pattern is all-or-nothing
sum |numeric| over ghost rows   : 0.0 <- every numeric field is exactly 0
rows with 0 vacancies elsewhere : 0
index span                      : 197478 - 606701
=== Step 2 - remove ghost rows ===
    3,988 structurally empty records dropped
               rows   cols    mem MB    NaN cells
BEFORE    1,048,585     22     401.7    1,092,453
AFTER     1,044,597     22     409.4    1,044,597
DELTA        -3,988     +0      +7.7      -47,856



{'rows': 1044597,
 'cols': 22,
 'mem': np.float64(409.4),
 'na': 1044597,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'occupationId',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'salary_type',
  'status_id',
  'status_jobStatus',
  'title',
  'average_salary']}

## 3 · Synthetic test rows

Rows whose `metadata_jobPostId` matches `RANDOM_JOB_*` are generated test data, not MCF postings:
the IDs embed a generation timestamp and the salaries are impossible. They are removed by **ID
pattern**, not by index, so the filter survives a reload or a re-sorted extract.

The `ATS-` prefixed rows are kept — they are a legitimate second source (real companies, sane
salaries) and get recorded in a new `source` column later.

In [23]:
synthetic = df['metadata_jobPostId'].str.match(SYNTHETIC_ID_RE, na=False)
prefix = df['metadata_jobPostId'].str.extract(r'^([A-Za-z_]+)')[0].value_counts()

print('ID prefixes present:')
print(prefix.to_string(), '\n')
print('synthetic rows to remove:', int(synthetic.sum()))
if synthetic.any():
    print(df.loc[synthetic, ['metadata_jobPostId', 'title', 'salary_minimum', 'salary_maximum']]
            .to_string(index=False))

metrics['synthetic'] = {'n': int(synthetic.sum()), 'prefixes': prefix.to_dict(),
                  'max_salary': int(df.loc[synthetic, 'salary_maximum'].max()) if synthetic.any() else 0}

before = snapshot(df)
df = df.loc[~synthetic].copy()
compare(before, df, 'Step 3 - remove synthetic test rows',
        f'{metrics["synthetic"]["n"]} RANDOM_JOB_* rows dropped')

ID prefixes present:
0
MCF            1044463
ATS                124
RANDOM_JOB_         10 

synthetic rows to remove: 10
               metadata_jobPostId                                                                        title  salary_minimum  salary_maximum
RANDOM_JOB_20251115011346015685_0                                                  Senior Manager - Operations          107908         6142101
RANDOM_JOB_20251115011346673349_1                                                           Resident Physician          108872        10734314
RANDOM_JOB_20251115011347120191_2 Senior Logistics Executive (1 yr contract) - up to $5k/West/SCIENCE MNC #HAO          276583         2720804
RANDOM_JOB_20251115011347466118_3                                                             Language Teacher          324072        20862169
RANDOM_JOB_20251115011347817248_4                                         Sales Associate (Home Audio, Retail)          262482        15531134
RANDOM_JOB_20251115

{'rows': 1044587,
 'cols': 22,
 'mem': np.float64(409.4),
 'na': 1044587,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'occupationId',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'salary_type',
  'status_id',
  'status_jobStatus',
  'title',
  'average_salary']}

## 4 · Prune dead columns

Detected, not hard-coded: a column is dead if it is entirely empty — every value `NaN` *or* a
blank string, since `''` is not `NaN` and would otherwise pass an `isna()` check — or if it holds a
single distinct value across a million real rows. A constant column cannot correlate with anything,
cannot be filtered on, and costs memory on every read.

The same rule is applied at the other end of the pipeline: `DEAD_COLUMNS` (Step 13) names the
`JOBS_SCHEMA` fields that would be constant or blank if they were materialised, so they are never
created in the first place.

`salary_type` being constant is information — it means *all salaries are monthly SGD* — so it is
recorded in the report rather than repeated a million times in the frame.

In [24]:
def all_blank(s):
    'True if every value is NaN or an empty / whitespace-only string - a column with no content.'
    # Categoricals are answered from their categories - fillna() on one raises rather than
    # reporting emptiness, and by Step 12 the text columns are categorical.
    if isinstance(s.dtype, pd.CategoricalDtype):
        return all(str(v).strip() == '' for v in s.cat.categories)
    # pandas 3 gives string columns a `str` dtype; pandas 2 leaves them `object`. Anything else
    # (numeric, bool, datetime) has no notion of blank.
    if not (s.dtype == object or pd.api.types.is_string_dtype(s)):
        return False
    return s.astype('object').fillna('').astype(str).str.strip().eq('').all()


empty_cols    = [c for c in df.columns if df[c].isna().all() or all_blank(df[c])]
constant_cols = [c for c in df.columns if c not in empty_cols and df[c].nunique(dropna=True) <= 1]
const_vals    = {c: df[c].dropna().iloc[0] if df[c].notna().any() else None for c in constant_cols}

print('empty    (all NaN or blank) :', empty_cols)
print('constant (1 value only)     :', {c: str(v) for c, v in const_vals.items()})

metrics['dead_cols'] = {'empty': empty_cols, 'constant': {c: str(v) for c, v in const_vals.items()}}

before = snapshot(df)
df = df.drop(columns=empty_cols + constant_cols)
compare(before, df, 'Step 4 - drop empty and zero-variance columns')

empty    (all NaN or blank) : ['occupationId']
constant (1 value only)     : {'salary_type': 'Monthly', 'status_id': '0'}
=== Step 4 - drop empty and zero-variance columns ===
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     22     409.4    1,044,587
AFTER     1,044,587     19     376.9            0
DELTA            +0     -3     -32.5   -1,044,587
    columns removed: ['occupationId', 'salary_type', 'status_id']



{'rows': 1044587,
 'cols': 19,
 'mem': np.float64(376.9),
 'na': 0,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary']}

## 5 · Type conversion — dates

`object`/`str` → `datetime64[ns]`. **Loss check:** the conversion is only accepted if
`to_datetime(...).dt.strftime('%Y-%m-%d')` reproduces the original string on every row. If that
assertion holds, the change is provably lossless.

Cross-field logic is validated at the same time — a repost cannot predate its original, and a
listing cannot expire before it is posted.

In [25]:
DATE_COLS = ['metadata_newPostingDate', 'metadata_originalPostingDate', 'metadata_expiryDate']

print('BEFORE')
print(df[DATE_COLS].dtypes.astype(str).to_string())
print(df[DATE_COLS].head(3).to_string(index=False), '\n')

date_rows = []
before = snapshot(df)
for c in DATE_COLS:
    original = df[c]
    parsed   = pd.to_datetime(original, errors='coerce')
    unparsed = int(parsed.isna().sum() - original.isna().sum())
    lossless = bool((parsed.dt.strftime('%Y-%m-%d') == original).all())
    assert unparsed == 0 and lossless, f'{c} would lose data - refusing to convert'
    df[c] = parsed
    date_rows.append({'column': c, 'from': str(original.dtype), 'to': str(parsed.dtype),
                      'unparseable': unparsed, 'round_trip_identical': lossless,
                      'min': str(parsed.min().date()), 'max': str(parsed.max().date())})

date_tbl = pd.DataFrame(date_rows)
metrics['md_dates'] = to_markdown(date_tbl, index=False)

d_new, d_orig, d_exp = df[DATE_COLS[0]], df[DATE_COLS[1]], df[DATE_COLS[2]]
metrics['date_checks'] = {'orig_after_new': int((d_orig > d_new).sum()),
                    'expiry_before_post': int((d_exp <= d_new).sum()),
                    'lifespan_median': float((d_exp - d_new).dt.days.median()),
                    'lifespan_max': int((d_exp - d_new).dt.days.max())}

print('AFTER')
print(df[DATE_COLS].dtypes.astype(str).to_string())
print(date_tbl.to_string(index=False), '\n')
print('originalPostingDate > newPostingDate :', metrics['date_checks']['orig_after_new'], '(violations)')
print('expiryDate <= newPostingDate         :', metrics['date_checks']['expiry_before_post'], '(violations)')
print('listing lifespan: median', metrics['date_checks']['lifespan_median'], 'days, max',
      metrics['date_checks']['lifespan_max'], 'days\n')
metrics['date_dtype'] = str(df[DATE_COLS[0]].dtype)
compare(before, df, 'Step 5 - parse date columns',
        f'str -> {metrics["date_dtype"]}, round-trip verified')

BEFORE
metadata_newPostingDate         str
metadata_originalPostingDate    str
metadata_expiryDate             str
metadata_newPostingDate metadata_originalPostingDate metadata_expiryDate
             2023-04-08                   2023-03-30          2023-05-08
             2023-04-08                   2023-04-08          2023-05-08
             2023-04-08                   2023-04-08          2023-04-22 

AFTER
metadata_newPostingDate         datetime64[us]
metadata_originalPostingDate    datetime64[us]
metadata_expiryDate             datetime64[us]
                      column from             to  unparseable  round_trip_identical        min        max
     metadata_newPostingDate  str datetime64[us]            0                  True 2023-03-28 2024-05-29
metadata_originalPostingDate  str datetime64[us]            0                  True 2022-10-03 2024-05-29
         metadata_expiryDate  str datetime64[us]            0                  True 2023-04-04 2024-06-28 

originalPostingDat

{'rows': 1044587,
 'cols': 19,
 'mem': np.float64(345.2),
 'na': 0,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary']}

## 6 · Type conversion — `categories` JSON → bridge table

`categories` holds a JSON array of `{"id", "category"}` objects. As a string it is unqueryable:
filtering "all IT jobs" needs a substring match that also catches the label inside other text.

The **many-to-many** structure is preserved in a separate `job_category` bridge table.
Keeping only the first category would be the lossy shortcut here — the notebook measures exactly
how many category assignments that would discard before deciding.

In [26]:
before = snapshot(df)
parsed_cat = df['categories'].map(json.loads)
n_per_row  = parsed_cat.map(len)

job_category = (pd.DataFrame({'metadata_jobPostId': df['metadata_jobPostId'], 'c': parsed_cat})
                  .explode('c', ignore_index=True))
job_category['category_id'] = job_category['c'].map(lambda d: d['id']).astype('int16')
job_category['category']    = job_category['c'].map(lambda d: d['category']).astype('category')
job_category = job_category.drop(columns='c')

df['primary_category'] = parsed_cat.map(lambda v: v[0]['category'])
df['n_categories']     = n_per_row.astype('int8')

metrics['categories'] = {
    'distinct': int(job_category['category'].nunique()),
    'assignments': len(job_category),
    'empty_arrays': int((n_per_row == 0).sum()),
    'per_row': {int(k): int(v) for k, v in n_per_row.value_counts().sort_index().items()},
    'multi_pct': round((n_per_row > 1).mean() * 100, 1),
    'lost_if_first_only': int(len(job_category) - len(df)),
}
metrics['md_top_categories'] = to_markdown(
    job_category['category'].value_counts().head(10).rename('postings').to_frame())

print('categories per posting :', metrics['categories']['per_row'])
print('distinct categories    :', metrics['categories']['distinct'])
print('total assignments      :', f'{metrics["categories"]["assignments"]:,}')
print(f'multi-category rows    : {metrics["categories"]["multi_pct"]}%  '
      f'-> keeping only the first would discard {metrics["categories"]["lost_if_first_only"]:,} assignments\n')
print('bridge table:')
print(job_category.head(4).to_string(index=False), '\n')

df = df.drop(columns='categories')
compare(before, df, 'Step 6 - normalise categories JSON',
        f'{len(job_category):,}-row bridge table extracted; raw JSON column dropped')

categories per posting : {1: 654951, 2: 208746, 3: 85461, 4: 38186, 5: 57243}
distinct categories    : 43
total assignments      : 1,767,785
multi-category rows    : 37.3%  -> keeping only the first would discard 723,198 assignments

bridge table:
metadata_jobPostId  category_id                    category
  MCF-2023-0252866           13        Environment / Health
  MCF-2023-0252866           25               Manufacturing
  MCF-2023-0252866           36 Sciences / Laboratory / R&D
  MCF-2023-0273977           21      Information Technology 

=== Step 6 - normalise categories JSON ===
    1,767,785-row bridge table extracted; raw JSON column dropped
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     19     345.2            0
AFTER     1,044,587     20     291.4            0
DELTA            +0     +1     -53.8           +0
    columns removed: ['categories']
    columns added  : ['primary_category', 'n_categories']



{'rows': 1044587,
 'cols': 20,
 'mem': np.float64(291.4),
 'na': 0,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories']}

## 7 · Data fixing — salaries

The floor and ceiling are **not treated symmetrically**, and that asymmetry is deliberate:

* **`SALARY_FLOOR` fixes corrupted data.** `$1/month` is not a low wage, it is a required-field
  placeholder, and `$10`–`$15` is an hourly rate typed into a monthly-only field. Neither is a
  real number at *any* resolution, so there is nothing to preserve — both bounds are **nulled**.
* **`SALARY_CEILING` is a statistical judgment, not a data-quality fix.** Whether a $150,000/month
  posting is a genuine C-suite role or a missing decimal depends on the question being asked — a
  mean/std view wants it excluded, a fraud-detection or "highest paid roles" view wants to see it.
  That decision belongs to analysis, not to this pipeline, so ceiling violations are **flagged, not
  nulled**: the raw values are left untouched and a `salary_flag == 'outlier'` marker is added so
  any downstream aggregate can choose to filter them out with one line, or choose not to.

This mirrors the general rule: cap what is definitely wrong at cleaning time; flag what is
*probably* wrong and let the analysis stage decide, since the right threshold there depends on
what's being measured.

Floor values are **nulled, not winsorised**: winsorising invents a number, `NaN` admits ignorance
and lets `mean()` skip the row. A quoted salary is a **range**, so the pair is the unit of
validity for the floor check — if either bound is a placeholder, both are nulled. Nulling only the
maximum would leave rows like `$1 – $600` behind, whose average of `$300.50` sits below the floor
the rule is supposed to enforce.

The plain 1.5×IQR rule is explicitly rejected below regardless of which stage would apply it — it
would delete tens of thousands of legitimate senior roles.

**One carve-out on the floor side.** `SALARY_FLOOR` was checked against the risk of nulling
genuine internship stipends before being set: internship salaries in this file sit above
`SALARY_FLOOR` at both the median and the 5th percentile — report section 3.1 quotes both — so
almost none are legitimately below it, but the rare ones that are shouldn't be silently destroyed. Rows tagged `Internship/Attachment` with both bounds in
`[INTERN_STIPEND_FLOOR, SALARY_FLOOR)` are **kept, not nulled**, and marked `low_stipend` rather
than `undisclosed`.

Requires nullable `Int32`, since a plain `int64` column cannot hold `NaN`.

In [27]:
before = snapshot(df)
sal_before = df[['salary_minimum', 'salary_maximum', 'average_salary']].describe(
    percentiles=[.01, .5, .99, .999]).round(1)
print('BEFORE\n', sal_before.to_string(), '\n')

# why not IQR: show what the textbook rule would have cost
q1, q3 = df['average_salary'].quantile([.25, .75])
iqr_fence = q3 + 3 * (q3 - q1)
metrics['salary_iqr'] = {'q1': float(q1), 'q3': float(q3), 'fence_3iqr': float(iqr_fence),
                   'rows_above_fence': int((df['average_salary'] > iqr_fence).sum())}
print(f'3xIQR upper fence would be ${iqr_fence:,.0f} and would null '
      f'{metrics["salary_iqr"]["rows_above_fence"]:,} rows - rejected as far too aggressive\n')

# measured again here: the answer changes once the synthetic rows are gone (see Step 11)
metrics['float32_before'] = float32_check(df['average_salary'])
print(f'float32 round-trip error on average_salary as-is: '
      f'{metrics["float32_before"]["max_error"]} '
      f'({metrics["float32_before"]["half_unrepresentable"]} unrepresentable .5 value(s))\n')

df['salary_minimum'] = df['salary_minimum'].astype('Int32')
df['salary_maximum'] = df['salary_maximum'].astype('Int32')

low_min    = df['salary_minimum'] < SALARY_FLOOR
low_max    = df['salary_maximum'] < SALARY_FLOOR
high_max   = df['salary_maximum'] > SALARY_CEILING
sentinel_1 = int(((df['salary_minimum'] == 1) & (df['salary_maximum'] == 1)).sum())
inverted   = int((df['salary_minimum'] > df['salary_maximum']).sum())

is_internship  = df['employmentTypes'] == 'Internship/Attachment'
intern_stipend = (is_internship
                  & low_min & low_max
                  & (df['salary_minimum'] >= INTERN_STIPEND_FLOOR)
                  & (df['salary_maximum'] >= INTERN_STIPEND_FLOOR))

below_floor = (low_min | low_max) & ~intern_stipend

metrics['salary_fix'] = {
    'floor': SALARY_FLOOR, 'ceiling': SALARY_CEILING,
    'intern_stipend_floor': INTERN_STIPEND_FLOOR,
    'low_min_n': int(low_min.sum()), 'low_max_n': int(low_max.sum()),
    'high_n': int(high_max.sum()),
    'min_only_n': int((low_min & ~low_max).sum()),
    'intern_stipend_n': int(intern_stipend.sum()),
    'sentinel_exactly_1': sentinel_1, 'inverted': inverted,
    'max_before': int(sal_before.loc['max', 'salary_maximum']),
    'low_by_employment': df.loc[low_min | low_max, 'employmentTypes'].value_counts().head(4).to_dict(),
}
print(f'salary_minimum < {SALARY_FLOOR:,}   : {metrics["salary_fix"]["low_min_n"]:,} rows  (nulled)')
print(f'salary_maximum < {SALARY_FLOOR:,}   : {metrics["salary_fix"]["low_max_n"]:,} rows  (nulled, '
      f'of which exactly $1-$1: {sentinel_1:,})')
print(f'   -> low minimum but a plausible maximum: {metrics["salary_fix"]["min_only_n"]:,} rows, '
      f'caught only by the pair rule')
print(f'salary_maximum > {SALARY_CEILING:,} : {metrics["salary_fix"]["high_n"]:,} rows  '
      f'(kept, flagged "outlier" - NOT nulled)')
print(f'internship stipend carve-out ({INTERN_STIPEND_FLOOR:,}-{SALARY_FLOOR:,}): '
      f'{metrics["salary_fix"]["intern_stipend_n"]:,} rows kept, flagged low_stipend')
print('salary_minimum > salary_maximum :', inverted, '(no swap correction needed)')
print('employment mix of the low group :', metrics['salary_fix']['low_by_employment'], '\n')

df.loc[below_floor, ['salary_minimum', 'salary_maximum']] = pd.NA
df['salary_flag'] = pd.Series(np.select(
    [below_floor,           intern_stipend, high_max],
    ['undisclosed', 'low_stipend',  'outlier'],
    default='ok',
), index=df.index).astype('category')
df['average_salary'] = ((df['salary_minimum'] + df['salary_maximum']) / 2).astype('Float64')

sal_after = df[['salary_minimum', 'salary_maximum', 'average_salary']].describe(
    percentiles=[.01, .5, .99, .999]).round(1)
metrics['md_salary_before'] = to_markdown(sal_before)
metrics['md_salary_after']  = to_markdown(sal_after)
metrics['salary_fix']['nulled'] = int(below_floor.sum())
metrics['salary_fix']['coverage_pct'] = round(df['salary_maximum'].notna().mean() * 100, 2)
metrics['salary_fix']['flag_counts'] = df['salary_flag'].value_counts().to_dict()

print('AFTER\n', sal_after.to_string(), '\n')
print(f'salaries nulled: {int(below_floor.sum()):,}  |  remaining coverage: '
      f'{metrics["salary_fix"]["coverage_pct"]}%')
print('salary_flag breakdown:', metrics['salary_fix']['flag_counts'])
compare(before, df, 'Step 7 - fix salary sentinels and outliers',
        f'{int(below_floor.sum()):,} salaries -> NaN; {metrics["salary_fix"]["intern_stipend_n"]} internship '
        f'stipends kept; average_salary recomputed; salary_flag built')

BEFORE
        salary_minimum  salary_maximum  average_salary
count       1044587.0       1044587.0       1044587.0
mean           3828.2          5630.8          4729.5
std            3104.3         27096.0         13882.9
min               1.0             1.0             1.0
1%              500.0          1000.0           800.0
50%            3000.0          4500.0          3800.0
99%           13000.0         20000.0         16750.0
99.9%         20000.0         35000.0         27470.7
max          350000.0      25330000.0      12666400.0 

3xIQR upper fence would be $13,300 and would null 24,520 rows - rejected as far too aggressive

float32 round-trip error on average_salary as-is: 0.0 (0 unrepresentable .5 value(s))

salary_minimum < 500   : 9,581 rows  (nulled)
salary_maximum < 500   : 7,124 rows  (nulled, of which exactly $1-$1: 1,804)
   -> low minimum but a plausible maximum: 2,457 rows, caught only by the pair rule
salary_maximum > 100,000 : 269 rows  (kept, flagged "outlier

{'rows': 1044587,
 'cols': 21,
 'mem': np.float64(287.2),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_flag']}

### `minimumYearsExperience`

Range check only. Values above `EXPERIENCE_MAX` are not physically possible, but they are a
handful of rows and are a poster-side typo rather than a pipeline defect — nulling them changes no
aggregate. **`0` is left untouched: it is a real value** meaning "no experience required".

In [28]:
before = snapshot(df)
years = df['minimumYearsExperience']
impossible = years > EXPERIENCE_MAX
metrics['experience'] = {'max_before': int(years.max()), 'impossible': int(impossible.sum()),
                   'zeros': int((years == 0).sum()), 'zeros_pct': round((years == 0).mean() * 100, 1),
                   'tail': years[years > 20].value_counts().sort_index().tail(6).to_dict()}
print('BEFORE  max =', metrics['experience']['max_before'], '| tail values:', metrics['experience']['tail'])

df['minimumYearsExperience'] = years.astype('Int16').mask(impossible, pd.NA)
print('AFTER   max =', int(df['minimumYearsExperience'].max()),
      f'| {metrics["experience"]["impossible"]} impossible values -> NaN')
compare(before, df, 'Step 7b - cap impossible experience values')

BEFORE  max = 88 | tail values: {59: 1, 61: 1, 62: 1, 76: 1, 87: 2, 88: 1}
AFTER   max = 88 | 0 impossible values -> NaN
=== Step 7b - cap impossible experience values ===
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     21     287.2       28,731
AFTER     1,044,587     21     282.0       28,731
DELTA            +0     +0      -5.2           +0



{'rows': 1044587,
 'cols': 21,
 'mem': np.float64(282.0),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_flag']}

## 8 · Data fixing — text normalisation

Three operations get lumped together as "normalisation", and they are treated differently here:

| operation | example | treatment | why |
|---|---|---|---|
| **repair** | `strip()`, collapse internal whitespace, drop zero-width characters | **in place** | `"  Chef "` carries no information `"Chef"` lacks — there is nothing to preserve. The changed-row count below is what makes the mutation auditable. |
| **re-key** | `lower()` | **not stored at all** | Case-folding destroys a real display form (`IT` vs `it`, `PhD`, `C++`), so it must not overwrite the column — but `title.str.lower()` reconstructs it in one line, so a second column would duplicate a value already on disk. |

`title` gets the repair in place. The fold is computed **where it is used** — as a local key in
Step 10 — and never added to the frame. It does real work there: without it `Software Engineer`
and `software engineer` count as different roles, and the duplicate check misses every pair whose
titles differ only by casing — Step 10 measures how many. What
survives into the saved data is `dup_group_id`, the *result* of the folded grouping, not the fold.

`postedCompany_name` gets the repair **plus an in-place `.upper()`**, and that is the one place this
notebook knowingly bends the rule above. The source is uppercase for all but a single entry, so the
fold merges exactly one pair — `Church of Our Saviour` into `CHURCH OF OUR SAVIOUR`. That is a real
display form being destroyed, so by the table it should earn its own column; it does not get one
because a separate `company_normalised` column cost real memory and gave the frame two names for
one concept, in order to merge that single pair. Normalising a lone outlier to the house style of
every other company is the better trade, and it leaves `postedCompany_name` as the join key.
Whitespace alone merges nothing here — every defect is an internal double space inside a name that
stays distinct.

**Zero-width characters are part of the repair**, because `\s` does not match them: U+200B, the BOM
and the word joiner render as nothing yet split otherwise-identical titles into separate values —
`'\u200bACCOUNTS ASSISTANT'` counts separately from `'ACCOUNTS ASSISTANT'` in every chart. The
joiners U+200C/U+200D are excluded from that class on purpose: U+200D is what holds an emoji
sequence together, and titles in this file use it — the cell below counts them — so stripping it
would corrupt content rather than clean it.

Legal-suffix variation (`PTE. LTD.` vs `PTE LTD`) is left alone deliberately: that is entity
resolution, not cleaning, and needs its own reviewed pass.

In [29]:
before = snapshot(df)
t_raw, c_raw = df['title'], df['postedCompany_name']

# Zero-width characters that `\s` misses (see markdown above for why the joiners are excluded).
# Built from codepoints, not `\u200b` escapes: pandas dispatches to RE2, which rejects `\u` --
# and it keeps characters that are by definition invisible out of the source.
INVISIBLE = '[' + ''.join(map(chr, [0x200B, 0x200E, 0x200F, 0x2060, 0xFEFF])) + ']'


def repair(s):
    'Drop zero-width characters, collapse whitespace runs, trim the ends.'
    return (s.str.replace(INVISIBLE, '', regex=True)
             .str.replace(r'\s+', ' ', regex=True)
             .str.strip())


# Changed-row counts are measured against the FULL repair, not just a strip(): most company
# defects are internal double spaces, which a leading/trailing test does not see.
title_repaired   = repair(t_raw)
company_repaired = repair(c_raw.str.upper())

metrics['text'] = {
    'title_ws_rows': int((t_raw != title_repaired).sum()),
    'title_invisible_rows': int(t_raw.str.contains(INVISIBLE, regex=True).sum()),
    'company_invisible_rows': int(c_raw.str.contains(INVISIBLE, regex=True).sum()),
    # U+200D is excluded from INVISIBLE; count what that exclusion protects
    'emoji_zwj_rows': int(t_raw.str.contains(chr(0x200D), regex=False).sum()),
    'emoji_zwj_titles': int(t_raw[t_raw.str.contains(chr(0x200D), regex=False)].nunique()),
    'title_distinct_raw': int(t_raw.nunique()),
    'company_ws_rows': int((c_raw != company_repaired).sum()),
    'company_distinct_raw': int(c_raw.nunique()),
}

df['title'] = title_repaired
df['postedCompany_name'] = company_repaired

# Measured here; computed where it is used (Step 10).
title_folded = df['title'].str.lower()

metrics['text'].update({
    'title_distinct_clean': int(df['title'].nunique()),
    'title_distinct_folded': int(title_folded.nunique()),
    'company_distinct_clean': int(df['postedCompany_name'].nunique()),
})
metrics['text']['title_collapsed'] = metrics['text']['title_distinct_raw'] - metrics['text']['title_distinct_folded']

print(f'title    : {metrics["text"]["title_ws_rows"]:,} rows changed by the repair '
      f'({metrics["text"]["title_invisible_rows"]:,} of them carried a zero-width character)')
print(f'           distinct  raw {metrics["text"]["title_distinct_raw"]:,}'
      f'  -> stripped {metrics["text"]["title_distinct_clean"]:,}'
      f'  -> case-folded {metrics["text"]["title_distinct_folded"]:,}'
      f'   ({metrics["text"]["title_collapsed"]:,} collapsed)')
print(f'company  : {metrics["text"]["company_ws_rows"]:,} rows changed by the repair')
print(f'           distinct  raw {metrics["text"]["company_distinct_raw"]:,}'
      f'  -> repaired in place {metrics["text"]["company_distinct_clean"]:,}'
      f'   ({metrics["text"]["company_distinct_raw"] - metrics["text"]["company_distinct_clean"]:,} collapsed)\n')
compare(before, df, 'Step 8 - normalise title and company text',
        'title and postedCompany_name repaired in place; no derived columns added')

title    : 21,934 rows changed by the repair (184 of them carried a zero-width character)
           distinct  raw 377,084  -> stripped 375,510  -> case-folded 364,753   (12,331 collapsed)
company  : 9,616 rows changed by the repair
           distinct  raw 53,151  -> repaired in place 53,150   (1 collapsed)

=== Step 8 - normalise title and company text ===
    title and postedCompany_name repaired in place; no derived columns added
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     21     282.0       28,731
AFTER     1,044,587     21     282.0       28,731
DELTA            +0     +0      +0.0           +0



{'rows': 1044587,
 'cols': 21,
 'mem': np.float64(282.0),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_flag']}

## 9 · Data filling

**Almost nothing is filled, on purpose.** After the ghost rows are gone there are no explicit
`NaN`s left except the salaries this notebook deliberately created. The real question is the
*implicit* missingness encoded as `0`, and for each such column the answer is to leave it and
document it:

| column | decision |
|---|---|
| `metadata_totalNumberJobApplication` | leave — genuinely ambiguous, cannot separate "no applicants" from "not tracked" |
| `metadata_totalNumberOfView` | leave — a 0-view / 0-application posting is internally consistent |
| `minimumYearsExperience` | leave — `0` is a real value for entry-level roles |
| `salary_*` | **never** median-impute; salary is the dependent variable — nulled above |

The zero rate for each is printed by the cell below and tabulated in report section 4.

Filling salaries by group median would manufacture the very distribution the dashboard exists to
measure, and would tighten variance so every confidence interval comes out wrong. Coverage is
reported alongside instead.

The zero-inflation claim for `minimumYearsExperience` is *tested* below, not asserted: if the
zeros were spread evenly across seniority they would be a default value, not a real one.

In [30]:
before = snapshot(df)
zero_cols = ['metadata_totalNumberJobApplication', 'metadata_totalNumberOfView',
             'minimumYearsExperience', 'numberOfVacancies', 'metadata_repostCount']
zero_tbl = pd.DataFrame({
    'zeros':   [int((df[c] == 0).sum()) for c in zero_cols],
    'zero_pct': [round((df[c] == 0).mean() * 100, 1) for c in zero_cols],
    'decision': ['leave - ambiguous', 'leave - ambiguous', 'leave - real value',
                 'n/a - min is 1', 'leave - real value'],
}, index=zero_cols)
print(zero_tbl.to_string(), '\n')
metrics['md_zeros'] = to_markdown(zero_tbl)

# test the "0 years means entry level" claim instead of assuming it
zero_exp_mix = (df.loc[df['minimumYearsExperience'] == 0, 'positionLevels']
                  .value_counts(normalize=True).mul(100).round(1).head(5))
overall_mix  = df['positionLevels'].value_counts(normalize=True).mul(100).round(1)
seniority_mix = pd.DataFrame({'zero_exp_%': zero_exp_mix,
                    'overall_%': overall_mix.reindex(zero_exp_mix.index)})
print('seniority mix of rows with 0 years required, vs the frame overall:')
print(seniority_mix.to_string(), '\n')
metrics['md_zero_exp_mix'] = to_markdown(seniority_mix)

# derived columns - added rather than filled
df['listing_days'] = (df['metadata_expiryDate'] - df['metadata_newPostingDate']).dt.days.astype('int16')
df['is_repost']    = df['metadata_repostCount'] > 0
df['source']       = df['metadata_jobPostId'].str.extract(r'^([A-Za-z]+)')[0]

metrics['derived'] = {'listing_days': 'expiryDate - newPostingDate',
                'is_repost': 'repostCount > 0',
                'source': 'ID prefix (MCF / ATS)',
                'salary_flag': "'ok' / 'undisclosed' / 'outlier' / 'low_stipend' - reason code from Step 7; "
                               "'outlier' rows keep their raw value (not nulled)",
                'average_salary': '(min + max) / 2, recomputed after Step 7',
                'primary_category': 'first element of the categories JSON',
                'n_categories': 'length of the categories JSON'}
metrics['source_mix'] = df['source'].value_counts().to_dict()
metrics['filled_cells'] = 0
print('source mix:', metrics['source_mix'])
compare(before, df, 'Step 9 - filling decisions and derived columns',
        '0 cells imputed; 3 derived columns added')

                                      zeros  zero_pct            decision
metadata_totalNumberJobApplication   656375      62.8   leave - ambiguous
metadata_totalNumberOfView           179121      17.1   leave - ambiguous
minimumYearsExperience               114451      11.0  leave - real value
numberOfVacancies                         0       0.0      n/a - min is 1
metadata_repostCount                1001862      95.9  leave - real value 

seniority mix of rows with 0 years required, vs the frame overall:
                   zero_exp_%  overall_%
positionLevels                          
Fresh/entry level        61.7       11.4
Non-executive            13.6       12.6
Executive                10.0       24.3
Junior Executive          8.2       16.0
Professional              4.5       10.7 

source mix: {'MCF': 1044463, 'ATS': 124}
=== Step 9 - filling decisions and derived columns ===
    0 cells imputed; 3 derived columns added
               rows   cols    mem MB    NaN cells
BEFORE 

{'rows': 1044587,
 'cols': 24,
 'mem': np.float64(296.6),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_flag',
  'listing_days',
  'is_repost',
  'source']}

## 10 · Duplicate rows

Duplication is tested at five levels of strictness, because "duplicate" means different things
depending on the question being asked.

The decisive evidence is at the bottom: for same-day identical postings, most groups have
**different view counts**. That means the platform served them as separate listings that
accumulated separate traffic — they are real distinct records, not a loading artefact. So they are
**flagged, not dropped**, and the choice is left to the analysis that consumes the data.

**The key is built locally; only the group *number* is stored.** The case-folded title lives as a
local Series in the cell below and is never added to the frame — `title` already carries the value,
and `title.str.lower()` rebuilds the fold in one line, so a stored copy would duplicate a column
already on disk and could drift out of sync with its input (the argument this notebook already
makes for recomputing `average_salary`). The fold still earns its place in the key — the cell below
measures how many extra duplicate rows it catches against a case-sensitive title, and the figure
lands in report section 5 rather than being restated here.

`dup_group_id` then stores the *outcome* of that grouping for a small fraction of the cost — the
cell below prints both figures — so downstream deduplication is `drop_duplicates('dup_group_id')`
rather than rebuilding a nine-column key by hand.

In [31]:
before = snapshot(df)

# A local key, never a column - see the markdown above.
title_key = df['title'].str.lower()

CONTENT = pd.DataFrame({
    'title':      title_key,
    'company':    df['postedCompany_name'],
    'employment': df['employmentTypes'],
    'level':      df['positionLevels'],
    'category':   df['primary_category'],
    'salary_min': df['salary_minimum'],
    'salary_max': df['salary_maximum'],
    'vacancies':  df['numberOfVacancies'],
})
SAME_DAY = CONTENT.assign(posted=df['metadata_newPostingDate'])

KEYS = {
    '1. exact duplicate row (all columns)':        df,
    '2. duplicate primary key (jobPostId)':        df[['metadata_jobPostId']],
    '3. identical content, any date':              CONTENT,
    '4. identical content, same posting date':     SAME_DAY,
    '5. company + title + posting date':           SAME_DAY[['company', 'title', 'posted']],
}
dup_tbl = pd.DataFrame(
    [{'key': k, 'duplicate_rows': int(v.duplicated().sum()),
      'pct': round(v.duplicated().mean() * 100, 2)} for k, v in KEYS.items()]
).set_index('key')
print(dup_tbl.to_string(), '\n')
metrics['md_duplicates'] = to_markdown(dup_tbl)
metrics['dup'] = {'exact': int(dup_tbl.iloc[0, 0]), 'pk': int(dup_tbl.iloc[1, 0]),
            'content': int(dup_tbl.iloc[2, 0]), 'same_day': int(dup_tbl.iloc[3, 0])}

# What the fold is worth: the same key with the raw `title`, and the difference in rows flagged.
folded_members = SAME_DAY.duplicated(keep=False)
cased_members = SAME_DAY.assign(title=df['title']).duplicated(keep=False)
fold_only = folded_members & ~cased_members

# Store the group NUMBER, not the nine columns that produced it.
group_id = SAME_DAY.groupby(list(SAME_DAY.columns), dropna=False, observed=True).ngroup()
df['dup_group_id']    = group_id.astype('int32')
df['dup_group_size']  = group_id.map(group_id.value_counts()).astype('int16')
df['is_same_day_dup'] = df['dup_group_size'] > 1

dup_rows = df[df['is_same_day_dup']]
view_spread = dup_rows.groupby('dup_group_id', observed=True)[
    ['metadata_totalNumberOfView', 'metadata_totalNumberJobApplication']].nunique()
metrics['dup'].update({
    'same_day_rows': int(len(dup_rows)),
    'same_day_groups': int(len(view_spread)),
    'groups_views_differ': int((view_spread['metadata_totalNumberOfView'] > 1).sum()),
    'groups_apps_differ': int((view_spread['metadata_totalNumberJobApplication'] > 1).sum()),
    'largest_group': int(df['dup_group_size'].max()),
    'behalf_in_dups': round(dup_rows['metadata_isPostedOnBehalf'].mean() * 100, 1),
    'behalf_baseline': round(df['metadata_isPostedOnBehalf'].mean() * 100, 1),
})
metrics['md_dup_agencies'] = to_markdown(
    dup_rows['postedCompany_name'].value_counts().head(5).rename('duplicate_rows').to_frame())

print(f'same-day duplicate rows      : {metrics["dup"]["same_day_rows"]:,} in '
      f'{metrics["dup"]["same_day_groups"]:,} groups (largest {metrics["dup"]["largest_group"]:,})')
print(f'groups where views differ    : {metrics["dup"]["groups_views_differ"]:,} / '
      f'{metrics["dup"]["same_day_groups"]:,}  <- proof these are distinct listings')
print(f'groups where apps differ     : {metrics["dup"]["groups_apps_differ"]:,}')
print(f'posted-on-behalf in dup rows : {metrics["dup"]["behalf_in_dups"]}% vs '
      f'{metrics["dup"]["behalf_baseline"]}% baseline  <- recruitment agencies')
print('\ntop agencies by duplicate rows:')
print(dup_rows['postedCompany_name'].value_counts().head(5).to_string(), '\n')

# What the key would have cost as a stored column, against what its result actually costs.
# index=False: a Series' memory_usage() otherwise adds the frame's 8 MB Int64 index to BOTH
# figures, which would overstate the id column by 3x and flatter the comparison.
metrics['dup'].update({
    'fold_extra_rows': int(fold_only.sum()),
    'fold_extra_groups': int(df.loc[fold_only, 'dup_group_id'].nunique()),
    'groups_total': int(df['dup_group_id'].nunique()),
    'key_mem': round(title_key.memory_usage(deep=True, index=False) / 1e6, 1),
    'dup_id_mem': round(df['dup_group_id'].memory_usage(deep=True, index=False) / 1e6, 1),
})
print(f'dup_group_id      : {metrics["dup"]["groups_total"]:,} groups, {metrics["dup"]["dup_id_mem"]} MB stored')
print(f'case-folded title : {metrics["dup"]["key_mem"]} MB, used as a local key and discarded '
      f'(rebuild with title.str.lower())')
print(f'the fold is worth  : {metrics["dup"]["fold_extra_rows"]:,} extra duplicate rows in '
      f'{metrics["dup"]["fold_extra_groups"]:,} groups vs a case-sensitive title\n')

compare(before, df, 'Step 10 - flag duplicates',
        '0 rows dropped; dup_group_id + dup_group_size + is_same_day_dup added, '
        'case-folded key not stored')


                                         duplicate_rows    pct
key                                                           
1. exact duplicate row (all columns)                  0   0.00
2. duplicate primary key (jobPostId)                  0   0.00
3. identical content, any date                   369889  35.41
4. identical content, same posting date           34878   3.34
5. company + title + posting date                 49619   4.75 

same-day duplicate rows      : 56,222 in 21,344 groups (largest 260)
groups where views differ    : 16,990 / 21,344  <- proof these are distinct listings
groups where apps differ     : 5,902
posted-on-behalf in dup rows : 37.8% vs 5.9% baseline  <- recruitment agencies

top agencies by duplicate rows:
postedCompany_name
RECRUITPEDIA PTE. LTD.               16044
HD MANPOWER CONSULTANTS PTE. LTD.     3519
57 EMPLOYMENT AGENCY PTE. LTD.        2977
MINDFLEX EDUCATION PTE. LTD.          2814
ORIENTAL EMPLOYMENT PTE. LTD.         2394 

dup_group_id      

{'rows': 1044587,
 'cols': 27,
 'mem': np.float64(303.9),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_flag',
  'listing_days',
  'is_repost',
  'source',
  'dup_group_id',
  'dup_group_size',
  'is_same_day_dup']}

## 11 · Type conversion — categorical and downcast

Left until last, so the category sets contain only surviving values and the integer ranges are
measured on cleaned data.

Every downcast is **asserted safe** against the observed range before it is applied — the loop
refuses rather than silently wrapping. `metadata_totalNumberOfView` and
`metadata_totalNumberJobApplication` get `int32` rather than the tighter `int16` their current
range allows: `int16` leaves only ~24k headroom, and a single viral posting in a future extract
would overflow it silently.

`average_salary` is the interesting case, and it shows why dtype decisions belong at the *end* of a
pipeline: whether `float32` is lossy depends on how much cleaning has already happened. The cell
below measures the round-trip error at three points and reports all of them rather than asserting
a verdict.

In [32]:
before = snapshot(df)
DOWNCAST = {'minimumYearsExperience': 'Int8', 'metadata_repostCount': 'int8',
            'numberOfVacancies': 'int16', 'metadata_totalNumberOfView': 'int32',
            'metadata_totalNumberJobApplication': 'int32', 'listing_days': 'int16',
            'n_categories': 'int8'}
CATEGORICAL = ['employmentTypes', 'positionLevels', 'status_jobStatus',
               'postedCompany_name', 'primary_category', 'source']

rows = []
for c, t in DOWNCAST.items():
    lo, hi = df[c].min(), df[c].max()
    int_limits = np.iinfo(np.dtype(t.lower()))
    fits = bool(lo >= int_limits.min and hi <= int_limits.max)
    assert fits, f'{c} range [{lo},{hi}] does not fit {t}'
    rows.append({'column': c, 'from': str(df[c].dtype), 'to': t,
                 'observed_range': f'{lo} - {hi}', 'headroom': f'{int_limits.max - hi:,}', 'safe': fits})
    df[c] = df[c].astype(t)

for c in CATEGORICAL:
    n = df[c].nunique()
    mem_b = df[c].memory_usage(deep=True) / 1e6
    df[c] = df[c].astype('category')
    rows.append({'column': c, 'from': 'str', 'to': 'category',
                 'observed_range': f'{n:,} distinct',
                 'headroom': f'{mem_b - df[c].memory_usage(deep=True)/1e6:,.1f} MB saved', 'safe': True})

type_tbl = pd.DataFrame(rows)
metrics['md_types'] = to_markdown(type_tbl, index=False)
print(type_tbl.to_string(index=False), '\n')

# float32 viability, measured rather than assumed - compare against the two earlier answers
metrics['float32'] = float32_check(df['average_salary'])
metrics['float32']['half_values'] = metrics['float32']['half_rows']
print('float32 round-trip error on average_salary, at three points in the pipeline:')
for label, key in [('raw file', 'float32_raw'), ('after synthetic rows removed', 'float32_before'),
                   ('after salary fix', 'float32')]:
    stage = metrics[key]
    print(f'   {label:<28}: {stage["max_error"]}  '
          f'({stage["half_unrepresentable"]} .5 value(s) above ${FLOAT32_HALF_LIMIT:,})')
print(f'   {metrics["float32"]["half_rows"]:,} rows carry a genuine .5 fraction')
print('   -> kept as Float64 anyway: the saving is only a few MB and float32 would silently')
print('      re-break if a future extract reintroduces large values\n')

compare(before, df, 'Step 11 - categorical dtypes and integer downcast')

                            column  from       to  observed_range      headroom  safe
            minimumYearsExperience Int16     Int8          0 - 88            39  True
              metadata_repostCount int64     int8           0 - 2           125  True
                 numberOfVacancies int64    int16         1 - 999        31,768  True
        metadata_totalNumberOfView int64    int32        0 - 8190 2,147,475,457  True
metadata_totalNumberJobApplication int64    int32        0 - 1342 2,147,482,305  True
                      listing_days int16    int16          1 - 30        32,737  True
                      n_categories  int8     int8           1 - 5           122  True
                   employmentTypes   str category      8 distinct 16.8 MB saved  True
                    positionLevels   str category      9 distinct 20.7 MB saved  True
                  status_jobStatus   str category      3 distinct 11.9 MB saved  True
                postedCompany_name   str category 53,1

{'rows': 1044587,
 'cols': 27,
 'mem': np.float64(162.7),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_flag',
  'listing_days',
  'is_repost',
  'source',
  'dup_group_id',
  'dup_group_size',
  'is_same_day_dup']}

## 12 · Final validation and save

In [33]:
final_profile = column_profile(df)
print(final_profile.to_string(), '\n')
metrics['md_profile_final'] = to_markdown(final_profile)
metrics['final'] = snapshot(df)

assert df['metadata_jobPostId'].is_unique,     'primary key is not unique'
assert df['title'].notna().all(),              'ghost rows survived'
assert not df['metadata_jobPostId'].str.match(SYNTHETIC_ID_RE).any(), 'synthetic rows survived'
ok_rows      = df['salary_flag'] == 'ok'
stipend_rows = df['salary_flag'] == 'low_stipend'
outlier_rows = df['salary_flag'] == 'outlier'
undisclosed_rows = df['salary_flag'] == 'undisclosed'
assert df.loc[ok_rows, 'salary_maximum'].between(SALARY_FLOOR, SALARY_CEILING).all(), 'salary max out of range'
assert df.loc[ok_rows, 'salary_minimum'].between(SALARY_FLOOR, SALARY_CEILING).all(), 'salary min out of range'
assert df.loc[stipend_rows, 'salary_maximum'].between(INTERN_STIPEND_FLOOR, SALARY_FLOOR).all(), 'stipend carve-out out of range'
assert (df.loc[stipend_rows, 'employmentTypes'] == 'Internship/Attachment').all(), 'stipend carve-out leaked outside internships'
assert (df.loc[outlier_rows, 'salary_maximum'] > SALARY_CEILING).all(), 'outlier flag set on a row that is not actually over the ceiling'
assert df.loc[outlier_rows, 'salary_maximum'].notna().all(), 'outlier rows were nulled instead of flagged'
assert undisclosed_rows.sum() == int(df['salary_maximum'].isna().sum()), 'nulled rows and undisclosed flag disagree'
assert (df['salary_minimum'].isna() == df['salary_maximum'].isna()).all(), 'salary bounds nulled apart'
assert (df['metadata_expiryDate'] > df['metadata_newPostingDate']).all(), 'date logic violated'

# No dead column may reach jobs_cleaned.parquet - neither carried through from the source nor
# reintroduced by a derived column. Checked by name, and by re-running the Step 4 test.
leaked = [c for c in DEAD_COLUMNS if c in df.columns]
assert not leaked, f'dead columns present in the cleaned frame: {leaked}'
still_dead = [c for c in df.columns if df[c].nunique(dropna=True) <= 1 or all_blank(df[c])]
assert not still_dead, f'zero-variance or blank columns survived to the cleaned frame: {still_dead}'
print('all post-conditions passed')
print(f'dead columns absent from jobs_cleaned : {list(DEAD_COLUMNS)}')

df.to_parquet(OUT_DIR / 'jobs_cleaned.parquet', index=False)
job_category.to_parquet(OUT_DIR / 'job_category.parquet', index=False)
metrics['outputs'] = {'jobs_cleaned.parquet': len(df), 'job_category.parquet': len(job_category)}

print(f'\nwrote {OUT_DIR/"jobs_cleaned.parquet"}      {len(df):,} rows x {df.shape[1]} cols')
print(f'wrote {OUT_DIR/"job_category.parquet"}   {len(job_category):,} rows')
print(f'\nraw {metrics["raw"]["rows"]:,} rows / {metrics["raw"]["mem"]:,.1f} MB  ->  '
      f'clean {metrics["final"]["rows"]:,} rows / {metrics["final"]["mem"]:,.1f} MB '
      f'({(metrics["final"]["mem"]/metrics["raw"]["mem"]-1)*100:+.0f}%)')

                                             dtype  na_count  na_pct  n_unique    zeros
employmentTypes                           category         0    0.00         8        0
metadata_expiryDate                 datetime64[us]         0    0.00       449        0
metadata_isPostedOnBehalf                     bool         0    0.00         2        0
metadata_jobPostId                             str         0    0.00   1044587        0
metadata_newPostingDate             datetime64[us]         0    0.00       429        0
metadata_originalPostingDate        datetime64[us]         0    0.00       603        0
metadata_repostCount                          int8         0    0.00         3  1001862
metadata_totalNumberJobApplication           int32         0    0.00       369   656375
metadata_totalNumberOfView                   int32         0    0.00      1543   179121
minimumYearsExperience                        Int8         0    0.00        47   114451
numberOfVacancies               

## 13 · Feature enrichment — align to the `src/pipeline/` schema

The enrichment logic lives in **`src/pipeline/feature_enrichment.py`**, not here — it is production
logic the pipeline (or any other consumer) should import rather than reimplement, and it is far
easier to test and review as a module. This notebook imports and applies it.

`feature_enrichment(df, job_category=None)` renames the raw MCF columns to the names the production
pipeline and `jobs` table use, and derives the feature columns `feature_engineer.py` adds, so the
cleaned data feeds the dashboard without a translation layer. `schema_report(enriched)` returns the
missing / extra column lists the assertions below use.

The module is the single source of truth for **which** columns: `RENAME_MAP` for the 15 renames and
`JOBS_SCHEMA_COLUMNS` for what is produced. Neither is re-tabulated here, so neither can drift out
of sync with the code that runs.

Five `JOBS_SCHEMA` columns are **not** materialised — `DEAD_COLUMNS` in the setup cell names them
with the reason each is dead, and prints them when that cell runs. Writing a constant down a million
rows is the dead weight Step 4 removes from the source side; it would be incoherent to prune
`salary_type` for being constant and then invent `salary_currency` two steps later. They are absent
from the module's `JOBS_SCHEMA_COLUMNS` rather than skipped at build time, so nothing can read the
list and rebuild them. Nothing downstream breaks: `DatabaseManager.insert_jobs` back-fills
`created_at` and `salary_currency` and filters to the columns actually present, and the three text
columns are nullable. The cell below asserts none reached the enriched frame; Step 12 asserts the
same of the cleaned frame.

> Editing the module while the kernel is live? Run `%load_ext autoreload` and `%autoreload 2`
> before the import cell, or restart the kernel — Python caches imported modules.

**Two deliberate deviations from the pipeline**, flagged rather than silent:

1. **`job_id` carries `metadata_jobPostId`** instead of a fresh UUID, so enriched rows still join
   back to the raw layer, to `job_category.parquet`, and to MCF. The pipeline's UUID makes that
   impossible.
2. **`competitiveness_score` divides by the 99th percentile**, not `max()` — whose denominator is
   dominated by the un-nulled outlier. Step 14 then drops the column anyway, for a reason no choice
   of denominator fixes.

**Two `feature_engineer.py` columns are not reproduced.** `days_posted` is `now() - posting_date`,
which measures when ingestion ran and would change the saved parquet on every execution;
`listing_days` is the deterministic equivalent and is kept. `is_growth_role` marks essentially every
role in the file (`count > median * 0.2`).

Of everything `feature_engineer.py` computes, only `seniority_years` reaches the `jobs` table. The
notebook's own provenance columns (`salary_flag`, `dup_group_size`, `is_same_day_dup`,
`dup_group_id`, `source`, `n_categories`, `listing_days`) are retained, so `jobs_enriched.parquet`
is a superset of the `jobs` table and a loader can select just the `JOBS_SCHEMA` columns.


In [34]:
import sys

# Make the repo root importable regardless of where the kernel was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'pipeline').is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.pipeline.feature_enrichment import (
    JOBS_SCHEMA_COLUMNS,
    RENAME_MAP,
    feature_enrichment,
    schema_report,
)

print('imported from', (REPO_ROOT / 'src' / 'pipeline' / 'feature_enrichment.py').relative_to(REPO_ROOT))

enriched = feature_enrichment(df, job_category=job_category)

check = schema_report(enriched)
assert not check['missing'], f"schema columns still missing: {check['missing']}"
leaked = [c for c in DEAD_COLUMNS if c in enriched.columns]
assert not leaked, f'dead columns reached the enriched frame: {leaked}'
assert not [c for c in enriched.columns if all_blank(enriched[c])], 'a blank column reached the enriched frame'
assert enriched['job_id'].is_unique, 'job_id must stay unique to serve as the primary key'
assert len(enriched) == len(df), 'enrichment changed the row count'

metrics['enrichment'] = {
    'renamed': len(RENAME_MAP),
    'schema_cols': len(JOBS_SCHEMA_COLUMNS),
    'dead_cols': dict(DEAD_COLUMNS),
    'extra_cols': check['extra'],
    'sub_sector_filled': int(enriched['sub_sector'].notna().sum()),
    'skills_found': int((enriched['skills'] != 'Not Specified').sum()),
    'skill_count_mean': round(float(enriched['skill_count'].mean()), 3),
    'comp_median': round(float(enriched['competitiveness_score'].median()), 2),
    'module': 'src/pipeline/feature_enrichment.py',
}
print(f'renamed {len(RENAME_MAP)} columns; all {len(JOBS_SCHEMA_COLUMNS)} JOBS_SCHEMA columns present')
print(f'dead columns, excluded from the schema list and never built ({len(DEAD_COLUMNS)}): '
      f'{list(DEAD_COLUMNS)}')
print(f"extra columns retained beyond JOBS_SCHEMA ({len(check['extra'])}): {check['extra']}\n")

print('--- JOBS_SCHEMA-aligned preview ---')
print(enriched[['job_id', 'title', 'company', 'sector', 'sub_sector', 'salary_min', 'salary_max',
                'experience_level', 'seniority_years', 'job_type']].head(4).to_string(index=False))
print()
print('derived feature columns:')
print(enriched[['salary_midpoint', 'salary_band', 'skills', 'skill_count',
                'listing_days', 'competitiveness_score']].head(4).to_string(index=False))
print()
print(f'sub_sector populated  : {metrics["enrichment"]["sub_sector_filled"]:,} rows '
      f'({metrics["enrichment"]["sub_sector_filled"]/len(enriched)*100:.1f}%) - pipeline leaves this NULL')
print(f'skills matched        : {metrics["enrichment"]["skills_found"]:,} rows '
      f'({metrics["enrichment"]["skills_found"]/len(enriched)*100:.1f}%), mean skill_count '
      f'{metrics["enrichment"]["skill_count_mean"]}')
print(f'competitiveness median: {metrics["enrichment"]["comp_median"]} '
      f'(pipeline max()-denominator would give ~0.02 for the salary half)\n')

compare(snapshot(df), enriched, 'Step 13 - feature enrichment',
        f'{len(RENAME_MAP)} columns renamed to the src/pipeline/ schema; '
        f'all {len(JOBS_SCHEMA_COLUMNS)} JOBS_SCHEMA columns satisfied, '
        f'{len(DEAD_COLUMNS)} dead ones excluded')

imported from src/pipeline/feature_enrichment.py
renamed 15 columns; all 23 JOBS_SCHEMA columns present
dead columns, excluded from the schema list and never built (5): ['location', 'salary_currency', 'description', 'requirements', 'created_at']
extra columns retained beyond JOBS_SCHEMA (11): ['metadata_isPostedOnBehalf', 'metadata_newPostingDate', 'status_jobStatus', 'n_categories', 'source', 'dup_group_size', 'is_same_day_dup', 'salary_band', 'skill_count', 'competitiveness_score', 'job_label']

--- JOBS_SCHEMA-aligned preview ---
          job_id                                                          title                    company                 sector    sub_sector  salary_min  salary_max experience_level  seniority_years  job_type
MCF-2023-0252866      Food Technologist - Clementi | Entry Level | Up to $2,800        WORKSTONE PTE. LTD.   Environment / Health Manufacturing        2000        2800      Entry Level                0 Permanent
MCF-2023-0273977 Software Engineer (F

{'rows': 1044587,
 'cols': 34,
 'mem': np.float64(218.2),
 'na': 683682,
 'columns': ['job_type',
  'expiry_date',
  'metadata_isPostedOnBehalf',
  'job_id',
  'metadata_newPostingDate',
  'posting_date',
  'repost_count',
  'applications',
  'views',
  'seniority_years',
  'vacancies',
  'position_level',
  'company',
  'salary_max',
  'salary_min',
  'status_jobStatus',
  'title',
  'salary_midpoint',
  'sector',
  'n_categories',
  'salary_flag',
  'listing_days',
  'is_repost',
  'source',
  'dup_group_id',
  'dup_group_size',
  'is_same_day_dup',
  'sub_sector',
  'experience_level',
  'salary_band',
  'skills',
  'skill_count',
  'competitiveness_score',
  'job_label']}

## 14 · Prune low-value derived columns

Step 4 pruned columns that were dead on arrival. This applies the same test to the columns *this
pipeline creates* — and two fail it. They are not constant, so Step 4's zero-variance rule does not
catch them; they fail a different test, **they carry no information about what they claim to
measure**, which is only visible once you check them against the data. The cell below measures each
one before dropping it, and the numbers land in report section 7.

* **`skill_count`** — a keyword match over job *titles*. `description` and `requirements` do not
  exist in the source, so the regex only ever sees a headline, and a headline is not where a
  posting lists its skills.
* **`competitiveness_score`** — meant to rank how contested a posting is, but barely correlated
  with applications, views or vacancies. A score that does not track its own subject is worse than
  absent, because a dashboard presents it as though it does.

Both are **computed and then dropped** rather than removed from `feature_enrichment.py`: that
module is shared production code, and `competitiveness_score` needs `skill_count` to build. This
step decides only what is worth *storing*.

`skills` is **kept** despite the same limitation and a heavier footprint — it is a `JOBS_SCHEMA`
column, so dropping it would break the loader contract. Read it as a headline-keyword signal, not a
requirements analysis.


In [35]:
before = snapshot(enriched)

PRUNE_TARGETS = {
    'skill_count':           'keyword-matched from titles; the source has no description text',
    'competitiveness_score': 'does not track its own inputs',
}
CONTEST = ['applications', 'views', 'vacancies']

rows = []
for c in PRUNE_TARGETS:
    s = enriched[c].astype('Float64')
    rows.append({
        'column': c,
        'distinct': int(enriched[c].nunique()),
        'zero_pct': round(float((s == 0).mean() * 100), 1),
        'MB': round(enriched[c].memory_usage(deep=True, index=False) / 1e6, 1),
        **{f'corr_{t}': round(float(s.corr(enriched[t].astype('Float64'))), 3) for t in CONTEST},
        'reason': PRUNE_TARGETS[c],
    })
prune_tbl = pd.DataFrame(rows).set_index('column')
print(prune_tbl.to_string(), '\n')
metrics['md_pruned'] = to_markdown(prune_tbl)

metrics['prune'] = {
    'columns': list(PRUNE_TARGETS),
    'mb': round(sum(r['MB'] for r in rows), 1),
    'skill_zero_pct': rows[0]['zero_pct'],
    'skill_distinct': rows[0]['distinct'],
    'comp_corr': {t: rows[1][f'corr_{t}'] for t in CONTEST},
    'skills_not_specified_pct': round(float((enriched['skills'] == 'Not Specified').mean() * 100), 1),
    'skills_distinct': int(enriched['skills'].nunique()),
    'skills_mb': round(enriched['skills'].memory_usage(deep=True, index=False) / 1e6, 1),
}
print(f'`skills` is KEPT ({metrics["prune"]["skills_distinct"]} distinct, '
      f'{metrics["prune"]["skills_not_specified_pct"]}% "Not Specified", {metrics["prune"]["skills_mb"]} MB) '
      f'- it is a JOBS_SCHEMA column, so dropping it would break the loader contract.\n')

enriched = enriched.drop(columns=list(PRUNE_TARGETS))

# the schema contract must still hold after the prune - these were provenance extras, not schema
check = schema_report(enriched)
assert not check['missing'], f"prune removed a JOBS_SCHEMA column: {check['missing']}"
metrics['enrichment']['extra_cols'] = check['extra']

compare(before, enriched, 'Step 14 - prune low-value derived columns',
        f'{metrics["prune"]["mb"]} MB dropped; all {len(JOBS_SCHEMA_COLUMNS)} JOBS_SCHEMA columns intact')

enriched.to_parquet(OUT_DIR / 'jobs_enriched.parquet', index=False)
metrics['outputs']['jobs_enriched.parquet'] = len(enriched)
print(f'wrote {OUT_DIR/"jobs_enriched.parquet"}   {len(enriched):,} rows x {enriched.shape[1]} cols')


                       distinct  zero_pct   MB  corr_applications  corr_views  corr_vacancies                                                           reason
column                                                                                                                                                        
skill_count                   7      97.1  1.0             -0.008       -0.01          -0.004  keyword-matched from titles; the source has no description text
competitiveness_score      3483       0.0  9.4              0.091        0.06          -0.031                                    does not track its own inputs 

`skills` is KEPT (297 distinct, 97.1% "Not Specified", 21.7 MB) - it is a JOBS_SCHEMA column, so dropping it would break the loader contract.

=== Step 14 - prune low-value derived columns ===
    10.4 MB dropped; all 23 JOBS_SCHEMA columns intact
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     34     218.2      683,682
AFTER     1,04

## 15 · Generate the markdown report

Assembles `docs/data-cleaning-report-generated.md` from the `M` metrics collected above. Prose
carries the justification; every number is interpolated from the run, so the report cannot drift
away from the data.

In [36]:
params = metrics['params']
# The value each dead column would have held, for the section 6 table.
DEAD_COL_VALUES = {'location': "`'Singapore'` on every row", 'salary_currency': "`'SGD'` on every row",
                   'description': '`\'\'` (empty string)', 'requirements': '`\'\'` (empty string)',
                   'created_at': '`Timestamp.now()` at run time'}
steps_tbl = to_markdown(pd.DataFrame([
    {'step': s['step'], 'rows': f'{s["rows_before"]:,} -> {s["rows_after"]:,}',
     'cols': f'{s["cols_before"]} -> {s["cols_after"]}',
     'mem MB': f'{s["mem_before"]:,.0f} -> {s["mem_after"]:,.0f}'} for s in metrics['steps']]), index=False)

lines = []
add = lines.append

add('# Data Cleaning Report - SGJobData')
add('')
add(f'Generated by `notebooks/data_cleaning.ipynb`. Source: `{RAW_CSV}`. '
  'Every figure is computed at run time from the data itself.')
add('')
add('> **What this document is.** The execution record: what the cleaning pipeline '
  'actually did, measured on the run that produced it. Do not hand-edit -- re-running '
  'the notebook overwrites it.')
add('>')
add('> **Companion document.** [`data-cleaning-decisions.md`](data-cleaning-decisions.md) '
  'carries the *reasoning* behind each choice below -- what was considered, what was '
  'rejected, and why. This report is authoritative for all figures.')
add('')
add(f'| | rows | cols | memory (MB) | NaN cells |')
add(f'| --- | ---: | ---: | ---: | ---: |')
add(f'| raw | {metrics["raw"]["rows"]:,} | {metrics["raw"]["cols"]} | {metrics["raw"]["mem"]:,.1f} | {metrics["raw"]["na"]:,} |')
add(f'| clean | {metrics["final"]["rows"]:,} | {metrics["final"]["cols"]} | {metrics["final"]["mem"]:,.1f} | {metrics["final"]["na"]:,} |')
add('')
add(f'**{metrics["raw"]["rows"] - metrics["final"]["rows"]:,} rows removed '
  f'({(1 - metrics["final"]["rows"]/metrics["raw"]["rows"])*100:.2f}%). '
  f'Memory {(metrics["final"]["mem"]/metrics["raw"]["mem"] - 1)*100:+.0f}%. '
  f'Cells imputed: {metrics["filled_cells"]}.**')
add('')
add('### Step ledger')
add('')
add(steps_tbl)
add('')
add('### Parameters')
add('')
add(f'| parameter | value | meaning |')
add(f'| --- | --- | --- |')
add(f'| `SALARY_FLOOR` | {params["SALARY_FLOOR"]:,} | monthly SGD at or below which a salary is a placeholder - **nulled** |')
add(f'| `SALARY_CEILING` | {params["SALARY_CEILING"]:,} | monthly SGD above which a salary is a statistical outlier - **flagged, not nulled** |')
add(f'| `INTERN_STIPEND_FLOOR` | {params["INTERN_STIPEND_FLOOR"]:,} | monthly SGD below which even an internship stipend is implausible |')
add(f'| `EXPERIENCE_MAX` | {params["EXPERIENCE_MAX"]} | years above which the value is impossible |')
add(f'| `SYNTHETIC_ID_RE` | `{params["SYNTHETIC_ID_RE"]}` | job-ID pattern marking generated test rows |')
add('')
add('---')
add('')

# ---- 1. type conversions -----------------------------------------------------
add('## 1. Column type conversions - and where data would be lost')
add('')
add(f'### 1.1 Dates: `str` -> `{metrics["date_dtype"]}`  *(lossless, asserted)*')
add('')
add(metrics['md_dates'])
add('')
add('**Justification.** As strings these columns cannot do date arithmetic, resampling, or `.dt` '
  'access - every time-series view in the dashboard needs real datetimes. **No data loss:** the '
  'notebook refuses the conversion unless `strftime` round-trips to the original string on every '
  'row, which it does. No timezone is applied, because the source has no time component and '
  'localising would imply precision that does not exist.')
add('')
add(f'Cross-field logic also holds: `originalPostingDate > newPostingDate` in '
  f'**{metrics["date_checks"]["orig_after_new"]}** rows, `expiryDate <= newPostingDate` in '
  f'**{metrics["date_checks"]["expiry_before_post"]}** rows. Median lifespan '
  f'{metrics["date_checks"]["lifespan_median"]:.0f} days.')
add('')
add('### 1.2 `categories` JSON -> bridge table  *(lossless as done; lossy if shortcut)*')
add('')
add(f'- {metrics["categories"]["distinct"]} distinct categories, '
  f'{metrics["categories"]["assignments"]:,} assignments over {metrics["final"]["rows"]:,} postings')
add('- categories per posting: ' + ', '.join(
    f'{k} -> {v:,} postings' for k, v in metrics['categories']['per_row'].items()))
add(f'- {metrics["categories"]["multi_pct"]}% of postings carry more than one category')
add('')
add(metrics['md_top_categories'])
add('')
add(f'**Justification.** A JSON string cannot be filtered on - "all IT jobs" would need a substring '
  f'match that also catches the label inside unrelated text. The many-to-many relationship is '
  f'preserved in `job_category.parquet` ({metrics["categories"]["assignments"]:,} rows). '
  f'**This is where loss would occur if shortcut:** keeping only the first category would discard '
  f'{metrics["categories"]["lost_if_first_only"]:,} assignments and systematically understate every '
  f'secondary category. `primary_category` is kept on the wide table as a convenience only.')
add('')
add('### 1.3 Categorical and integer conversions  *(lossless, range-asserted)*')
add('')
add(metrics['md_types'])
add('')
add('**Justification.** Eight distinct employment types over a million rows is the textbook case for '
  '`category`: the string is stored once and each row holds a small integer code. Every integer '
  'downcast is asserted against the observed range before being applied, so the loop fails loudly '
  'rather than wrapping silently. The two `metadata_total*` counters are given `int32` rather than '
  'the tighter `int16` their current range permits - `int16` leaves only ~24k headroom and one '
  'viral posting in a future extract would overflow it.')
add('')
add('### 1.4 `average_salary` -> `float32`: the conversion whose answer depends on when you ask')
add('')
add(f'| measured at | `.5` values above ${FLOAT32_HALF_LIMIT:,} | max round-trip error |')
add(f'| --- | ---: | ---: |')
add(f'| raw file | {metrics["float32_raw"]["half_unrepresentable"]} | **{metrics["float32_raw"]["max_error"]}** |')
add(f'| after synthetic rows removed | {metrics["float32_before"]["half_unrepresentable"]} | '
  f'{metrics["float32_before"]["max_error"]} |')
add(f'| after the Step 7 salary fix | {metrics["float32"]["half_unrepresentable"]} | '
  f'{metrics["float32"]["max_error"]} |')
add('')
if metrics['float32_raw']['max_error'] > 0:
    add(f'**On the raw file the cast is lossy** - error {metrics["float32_raw"]["max_error"]}, and it '
      f'traces to a single row. float32 has a 24-bit mantissa, so above ${FLOAT32_HALF_LIMIT:,} it '
      f'cannot represent a `.5` at all, and the raw file holds exactly '
      f'{metrics["float32_raw"]["half_unrepresentable"]} such value - a synthetic posting at '
      f'${metrics["float32_raw"]["worst_value"]:,.1f}. That single row is the whole of the error. The `.5` fractions themselves are '
      f'ordinary: {metrics["float32"]["half_values"]:,} surviving rows carry one (odd min+max) and '
      f'every one round-trips exactly, because they sit below the limit. Remove the synthetic rows '
      f'and the cast becomes lossless.')
    add('')
    add('The point is not that float32 is unusable - it is that "is this conversion lossless?" has '
      'no answer independent of the cleaning order. Profiling dtypes on a dirty frame gives a '
      'different verdict than profiling the same frame after junk-row removal, which is why every '
      'dtype decision in this notebook is made last.')
else:
    add(f'The cast is exact at every stage on this extract, including the raw file '
      f'({metrics["float32"]["half_values"]:,} rows carry a genuine `.5` fraction, all within float32 '
      f'precision).')
add('')
add(f'It is **kept as `Float64` regardless**: the saving is only a few MB, and float32 would '
  f'silently re-break the moment a future extract reintroduces large values. `average_salary` is '
  f'also fully derived from `salary_minimum`/`salary_maximum`, so it is recomputed after the '
  f'salary fixes rather than trusted - a stored derived column drifts out of sync with its inputs.')
add('')
add('### 1.5 Columns dropped as dead')
add('')
add(f'| column | reason |')
add(f'| --- | --- |')
for c in metrics['dead_cols']['empty']:
    add(f'| `{c}` | 100% NaN, 0 distinct values - empty column |')
for c, v in metrics['dead_cols']['constant'].items():
    add(f'| `{c}` | single value `{v}` on every row - zero variance |')
add('')
add('**Justification.** A constant column cannot correlate with anything and cannot be filtered on '
  'meaningfully. `salary_type` being constant is itself information - *all salaries are monthly '
  'SGD* - and belongs in this report rather than repeated a million times in the frame. '
  '`metadata_isPostedOnBehalf` is retained despite its 94/6 split: low variance is not zero '
  'variance, and it flags recruiter-posted listings, which matters for the duplicate analysis in '
  'section 5.')
add('')
add('---')
add('')

# ---- 2. ghost rows -----------------------------------------------------------
add('## 2. Ghost rows removed')
add('')
add(f'**{metrics["ghost"]["n"]:,} rows ({metrics["ghost"]["pct"]}% of the file) - all removed.**')
add('')
add(f'| evidence | value |')
add(f'| --- | --- |')
add(f'| rows where all {metrics["ghost"]["text_cols"]} text columns are NaN | {metrics["ghost"]["n"]:,} |')
add(f'| rows with a *partial* NaN pattern | **{metrics["ghost"]["partial"]}** |')
add(f'| observed NaN-count-per-row values | {metrics["ghost"]["bimodal"]} |')
add(f'| sum of abs() over every numeric column on these rows | **{metrics["ghost"]["numeric_abs_sum"]}** |')
add(f'| rows with 0 vacancies outside this group | {metrics["ghost"]["vacancies_zero_elsewhere"]} |')
add(f'| index span | {metrics["ghost"]["idx_min"]:,} - {metrics["ghost"]["idx_max"]:,} |')
add('')
add('**Justification for deletion rather than imputation.** The per-row NaN count is strictly '
  'bimodal - a row is either fully populated or fully blank, never in between - and every numeric '
  'field on the blank rows is exactly zero. There is nothing to impute *from*: no title, no '
  'company, no ID, no dates. Every real posting has at least one vacancy; a record with zero '
  'vacancies, zero salary and no identifier is not a job. Imputing would fabricate '
  f'{metrics["ghost"]["n"]:,} synthetic postings and inflate every count on the dashboard. At '
  f'{metrics["ghost"]["pct"]}% of the data, dropping costs nothing statistically.')
add('')
add('**Ordering trap.** `occupationId` is 100% NaN, so on the raw frame `df.dropna()` returns an '
  '**empty** DataFrame. The notebook uses an explicit all-text-NaN mask instead, which states the '
  'intent and cannot misfire.')
add('')
add(f'### Synthetic test rows: {metrics["synthetic"]["n"]} more removed')
add('')
add(f'Rows matching `{params["SYNTHETIC_ID_RE"]}` are generated test data - the IDs embed a generation '
  f'timestamp and salaries reach ${metrics["synthetic"]["max_salary"]:,}/month. Removed by ID pattern, '
  f'not by index, so the filter survives a reload. ID prefixes present in the raw file: '
  f'{metrics["synthetic"]["prefixes"]}. The `ATS-` rows are legitimate (real companies, sane salaries) '
  f'and are retained, tagged via the new `source` column: {metrics["source_mix"]}.')
add('')
add('---')
add('')

# ---- 3. data fixing ----------------------------------------------------------
add('## 3. Data fixing, per column')
add('')
add('### 3.1 `salary_minimum` / `salary_maximum` / `average_salary`')
add('')
add('**Before**')
add('')
add(metrics['md_salary_before'])
add('')
add('**After**')
add('')
add(metrics['md_salary_after'])
add('')
add(f'**The floor and ceiling are handled differently on purpose** - not two symmetric guard rails, '
  f'but two different kinds of problem:')
add('')
add(f'| defect | rows | treatment |')
add(f'| --- | ---: | --- |')
add(f'| `salary_minimum < {params["SALARY_FLOOR"]:,}` (undisclosed placeholder) | '
  f'{metrics["salary_fix"]["low_min_n"]:,} | **null** the pair |')
add(f'| `salary_maximum < {params["SALARY_FLOOR"]:,}` (undisclosed placeholder) | '
  f'{metrics["salary_fix"]["low_max_n"]:,} | **null** the pair |')
add(f'| of which exactly `$1 - $1` | {metrics["salary_fix"]["sentinel_exactly_1"]:,} | **null** the pair |')
add(f'| low minimum but plausible maximum (e.g. `$1 - $600`) | '
  f'{metrics["salary_fix"]["min_only_n"]:,} | **null** the pair |')
add(f'| internship, both bounds in `[{params["INTERN_STIPEND_FLOOR"]:,}, {params["SALARY_FLOOR"]:,})` | '
  f'{metrics["salary_fix"]["intern_stipend_n"]:,} | **keep**, flag `low_stipend` |')
add(f'| `salary_maximum > {params["SALARY_CEILING"]:,}` (statistical outlier) | '
  f'{metrics["salary_fix"]["high_n"]:,} | **keep, do not null** - flag `outlier` |')
add(f'| `salary_minimum > salary_maximum` | {metrics["salary_fix"]["inverted"]} | none needed |')
add('')
add(f'**Why the asymmetry.** `$1/month` and a `$10`-`$15` hourly rate typed into a monthly-only '
  f'field are not real numbers at *any* resolution - there is nothing to preserve, so the floor '
  f'defect is nulled. A `${params["SALARY_CEILING"]:,}+`/month posting is different: it might be a '
  f'genuine C-suite salary or it might be a missing decimal, and which one is true depends on the '
  f'question being asked of the data - a mean/std view wants it excluded, a "highest-paid roles" '
  f'or fraud-detection view wants to see it. That is an analysis-stage judgment, not a '
  f'cleaning-stage fact, so the raw value is left untouched and only a `salary_flag == \'outlier\'` '
  f'marker is added. Whether to filter on it is left to whoever runs the query.')
add('')
add(f'**The pair is the unit of validity for the floor check.** A quoted salary is a *range*, so if '
  f'either bound is a placeholder the whole range is untrustworthy. Nulling only `salary_maximum` '
  f'would leave {metrics["salary_fix"]["min_only_n"]:,} rows like `$1 - $600` behind, whose average of '
  f'$300.50 sits below the very floor the rule is meant to enforce. Both bounds are therefore '
  f'nulled together, and a post-condition asserts they are never nulled apart.')
add('')
add(f'**Justification for the floor.** $1/month is not a wage - it is what a poster enters to satisfy '
  f'a required field. Singapore has no statutory minimum wage, so there is no bright line; '
  f'${params["SALARY_FLOOR"]:,} is a conservative "cannot be a real monthly wage" threshold well below '
  f'the Progressive Wage Model floors (~$1,400-1,600). Note the low group skews part-time and '
  f'contract ({metrics["salary_fix"]["low_by_employment"]}), so some are hourly rates forced into a '
  f'monthly-only field - mis-unit rather than undisclosed, but both must be excluded from salary '
  f'aggregates.')
add('')
add(f'**Internship carve-out.** A flat ${params["SALARY_FLOOR"]:,} floor risked deleting genuine low '
  f'stipends, so it was checked before being applied rather than assumed. Internship salaries in '
  f'this file have a median of $1,200 and a 5th percentile of $800 - almost none are legitimately '
  f'this low - but {metrics["salary_fix"]["intern_stipend_n"]:,} rows tagged `Internship/Attachment` '
  f'with both bounds in `[{params["INTERN_STIPEND_FLOOR"]:,}, {params["SALARY_FLOOR"]:,})` are real stipends, '
  f'not placeholders. They are kept and marked `low_stipend` instead of `undisclosed`, visible to '
  f'any analysis that wants to exclude them, invisible to one that does not - the same '
  f'flag-don\'t-null principle used for the ceiling, applied at the low end.')
add('')
add(f'**Why not the textbook IQR rule** for a ceiling, if one were applied at cleaning time at all. '
  f'Q1=${metrics["salary_iqr"]["q1"]:,.0f}, Q3=${metrics["salary_iqr"]["q3"]:,.0f}, so even the lenient 3xIQR '
  f'upper fence sits at ${metrics["salary_iqr"]["fence_3iqr"]:,.0f} and would catch '
  f'{metrics["salary_iqr"]["rows_above_fence"]:,} rows - tens of thousands of legitimate senior roles. '
  f'Whatever threshold an analysis chooses to filter `salary_flag == \'outlier\'` on, it should not '
  f'be the IQR rule.')
add('')
add(f'Total nulled: **{metrics["salary_fix"]["nulled"]:,}** rows (floor only). Remaining salary coverage: '
  f'**{metrics["salary_fix"]["coverage_pct"]}%** - higher than if the ceiling were also nulled, because '
  f'those {metrics["salary_fix"]["high_n"]:,} rows are still present, just flagged. `average_salary` is '
  f'recomputed from the cleaned inputs rather than inherited, which means it **still contains the '
  f'flagged outliers** (max ${metrics["salary_fix"]["max_before"]:,}) until an analysis filters them out - '
  f'that is the point, not an oversight. A single reason code is built rather than a bare NaN: '
  f'`salary_flag` (`{metrics["salary_fix"]["flag_counts"]}`) records *why* a value is missing or '
  f'suspect - `undisclosed`, `outlier`, or the retained `low_stipend` carve-out - so the reason is '
  f'never lost along with the value. Filter with `salary_max.notna()`, or on the flag when the '
  f'reason matters.')
add('')
add('### 3.2 `minimumYearsExperience`')
add('')
add(f'Observed max was **{metrics["experience"]["max_before"]}** years; tail values '
  f'{metrics["experience"]["tail"]}. Values above {params["EXPERIENCE_MAX"]} are physically impossible and '
  f'were set to NaN - **{metrics["experience"]["impossible"]} rows**, too few to move any aggregate. '
  f'These are poster-side typos, not a pipeline defect, which is why the column is capped rather '
  f'than rebuilt. `0` is left untouched (see section 4).')
add('')
add('### 3.3 `title` and `postedCompany_name`')
add('')
add(f'| fix | before | after |')
add(f'| --- | ---: | ---: |')
add(f'| rows changed by the `title` repair | {metrics["text"]["title_ws_rows"]:,} | 0 |')
add(f'| ...of those, carrying a zero-width character | {metrics["text"]["title_invisible_rows"]:,} | 0 |')
add(f'| rows changed by the `postedCompany_name` repair | {metrics["text"]["company_ws_rows"]:,} | 0 |')
add(f'| distinct titles | {metrics["text"]["title_distinct_raw"]:,} | '
  f'{metrics["text"]["title_distinct_folded"]:,} (case-folded) |')
add(f'| distinct companies | {metrics["text"]["company_distinct_raw"]:,} | '
  f'{metrics["text"]["company_distinct_clean"]:,} (`postedCompany_name`, repaired in place) |')
add('')
add('**Justification.** Three operations get lumped together as "normalisation" and are treated '
  'differently here. A **repair** - stripping whitespace, collapsing internal runs, dropping '
  'zero-width characters - goes **in place**: `"  Chef "` carries no information `"Chef"` lacks, '
  'so there is nothing to preserve, and the changed-row counts above are what make the mutation '
  'auditable. A **re-key** - case-folding - gets its **own column**, because it destroys a '
  'display form (`IT` vs `it`, `PhD`, `C++`) that the dashboard needs.')
add('')
add(f'**Zero-width characters are part of the repair**, because `\\s` does not match them. U+200B, '
  f'the BOM and the word joiner render as nothing yet split otherwise-identical titles into '
  f'separate values - `ACCOUNTS ASSISTANT` prefixed with a zero-width space counts separately '
  f'from `ACCOUNTS ASSISTANT` in every chart. The joiners U+200C/U+200D are excluded from that '
  f'class deliberately: U+200D is what holds an emoji sequence together, and '
  f'{metrics["text"]["emoji_zwj_rows"]:,} rows ({metrics["text"]["emoji_zwj_titles"]:,} distinct titles) use '
  f'one, so stripping it would corrupt content rather than clean it.')
add('')
add(f'`title` gets the repair in place. The fold is **not stored at all** - not as an overwrite, '
  f'because it destroys the display form, and not as a second column, because `title.str.lower()` '
  f'rebuilds it in one line from a value already on disk. It is computed as a local key where it '
  f'is used (section 5) and discarded. It earns its place there: it collapses '
  f'**{metrics["text"]["title_collapsed"]:,}** distinct titles - without it "Software Engineer" and '
  f'"software engineer" count as different roles, and the duplicate check misses '
  f'{metrics["dup"]["fold_extra_rows"]:,} rows - and what survives into the saved data is '
  f'`dup_group_id`, the result of the folded grouping.')
add('')
add(f'`postedCompany_name` gets the repair **plus an in-place `.upper()`** - the one place this '
  f'pipeline knowingly bends the rule above. The source is uppercase for all but a single entry, '
  f'so the fold merges exactly one pair: `Church of Our Saviour` into `CHURCH OF OUR SAVIOUR`. '
  f'That is a real display form being destroyed, so strictly it should earn its own column. It '
  f'does not get one because an earlier `company_normalised` cost 6.1 MB and a second name for '
  f'one concept in order to merge that single pair; normalising a lone outlier to the house style '
  f'of the other {metrics["text"]["company_distinct_clean"] - 1:,} is the better trade, and it leaves '
  f'`postedCompany_name` as the join key. Whitespace alone merges nothing here - all '
  f'{metrics["text"]["company_ws_rows"]:,} defects are internal double spaces inside names that stay '
  f'distinct.')
add('')
add('Legal-suffix variation (`PTE. LTD.` vs `PTE LTD`) is left alone deliberately: that is entity '
  'resolution, not cleaning, and needs its own reviewed pass.')
add('')
add('### 3.4 Columns needing no fix')
add('')
add(f'`categories` parses cleanly on every row with {metrics["categories"]["empty_arrays"]} empty arrays. '
  f'`numberOfVacancies` spans 1-999 with no zeros; 999 looks like a UI cap rather than a true '
  f'count, which is worth a footnote on any "total vacancies" headline but is not a defect to '
  f'repair. `status_jobStatus`, `employmentTypes` and `positionLevels` have small clean value sets.')
add('')
add('---')
add('')

# ---- 4. data filling ---------------------------------------------------------
add('## 4. Data filling, per column')
add('')
add(f'**{metrics["filled_cells"]} cells were imputed.** After ghost-row removal the frame has no explicit '
  f'NaN outside the salaries this pipeline deliberately created. The filling question is really '
  f'about *implicit* missingness encoded as `0`:')
add('')
add(metrics['md_zeros'])
add('')
add('### Why nothing is filled')
add('')
add(f'**`metadata_totalNumberJobApplication`** - zero is genuinely bimodal: it means "nobody applied" '
  f'for many listings and "the counter was not populated at scrape time" for others, and the data '
  f'offers no way to separate them. Mean-filling would fabricate applications; NaN-ing all of them '
  f'would discard the majority of the column. Left as `0`, flagged here, and application-rate '
  f'metrics should be restricted to rows with views > 0.')
add('')
add(f'**`metadata_totalNumberOfView`** - same ambiguity at a lower rate. A posting with 0 views and 0 '
  f'applications is internally consistent, so view-based analysis on the non-zero subset is '
  f'defensible. Guard the division: `applications / views` is undefined for these rows.')
add('')
add(f'**`minimumYearsExperience`** - `0` is a real value meaning "no prior experience required". '
  f'This was tested rather than assumed - the seniority mix of the zero-experience rows against '
  f'the frame overall:')
add('')
add(metrics['md_zero_exp_mix'])
add('')
add('The zeros concentrate at the junior end rather than spreading evenly across seniority, which is '
  'what a defaulted-to-zero field would look like. They are real.')
add('')
add(f'**Salaries** - the ~{metrics["salary_fix"]["nulled"]:,} nulled values (the floor defect only) are '
  f'**not** imputed by median, group median, or regression. Salary is the dependent variable this '
  f'dashboard exists to measure; filling it with a group median manufactures the very distribution '
  f'being observed and tightens variance so every confidence interval comes out wrong. Exclude from '
  f'aggregates and report the {metrics["salary_fix"]["coverage_pct"]}% coverage rate alongside every '
  f'salary figure. If a downstream model cannot accept NaN, add an explicit `salary_imputed` flag '
  f'so the imputation is never invisible.')
add('')
add(f'The {metrics["salary_fix"]["high_n"]:,} ceiling outliers are a related but separate case: they are '
  f'not missing, so there is nothing to fill. `salary_flag == \'outlier\'` marks them and leaves '
  f'the decision to exclude, cap, or report them separately to whichever analysis consumes the '
  f'data - see section 3.1 for why that choice is deferred rather than made here.')
add('')
add('### Derived columns added instead of filling')
add('')
add('| column | definition |')
add('| --- | --- |')
for k, v in metrics['derived'].items():
    add(f'| `{k}` | {v} |')
add('')
add('---')
add('')

# ---- 5. duplicates -----------------------------------------------------------
add('## 5. Duplicate rows')
add('')
add(metrics['md_duplicates'])
add('')
add(f'**No rows were dropped as duplicates.** They are flagged via `dup_group_size` and '
  f'`is_same_day_dup` so the decision belongs to the analysis that consumes the data.')
add('')
add('### Justification')
add('')
add(f'**Keys 1 and 2 are clean.** {metrics["dup"]["exact"]} exact duplicate rows and {metrics["dup"]["pk"]} '
  f'duplicate `metadata_jobPostId` - the primary key is sound and the loader is not '
  f'double-reading anything. Any deduplication beyond this point is a judgement about business '
  f'meaning, not a repair.')
add('')
add(f'**Key 3 ({metrics["dup"]["content"]:,} rows) is serial reposting, not duplication.** The same role '
  f'posted by the same agency on different dates with different job IDs is how recruitment '
  f'agencies work. Each is a real posting that really appeared on the platform. Collapsing them '
  f'would erase the time dimension of hiring demand - exactly what a job-market dashboard is '
  f'measuring.')
add('')
add(f'**Key 4 ({metrics["dup"]["same_day"]:,} rows) is the only arguable case - and the evidence says '
  f'keep.** These are {metrics["dup"]["same_day_rows"]:,} rows in {metrics["dup"]["same_day_groups"]:,} groups '
  f'of identical same-day postings (largest group: {metrics["dup"]["largest_group"]:,}). The decisive '
  f'test: **{metrics["dup"]["groups_views_differ"]:,} of {metrics["dup"]["same_day_groups"]:,} groups have '
  f'different view counts** across their members, and {metrics["dup"]["groups_apps_differ"]:,} have '
  f'different application counts. Distinct traffic means the platform served them as separate '
  f'listings that jobseekers found and viewed separately. Dropping them would discard real '
  f'engagement data.')
add('')
add(f'They are also concentrated: {metrics["dup"]["behalf_in_dups"]}% are `isPostedOnBehalf` against a '
  f'{metrics["dup"]["behalf_baseline"]}% baseline, and the top contributors are manpower agencies.')
add('')
add(metrics['md_dup_agencies'])
add('')
add('**Recommended handling downstream.** Keep every row for volume and time-series work. When '
  'ranking employers or estimating distinct job openings, deduplicate at query time with '
  '`drop_duplicates(\'dup_group_id\')` - reversible, and the choice stays visible in the analysis '
  'rather than being baked into the stored dataset. `numberOfVacancies` is tracked separately, so '
  'a company posting five identical roles is not the same as one posting with five vacancies.')
add('')
add(f'`dup_group_id` stores the key-4 grouping as a group *number* rather than as the nine columns '
  f'that produced it: {metrics["dup"]["groups_total"]:,} groups in {metrics["dup"]["dup_id_mem"]} MB, against '
  f'the {metrics["dup"]["key_mem"]} MB the case-folded title component alone would cost as a stored '
  f'column. That fold is built as a local key and discarded - `title.str.lower()` rebuilds it in '
  f'one line - while the grouping it produces is preserved exactly, including the '
  f'{metrics["dup"]["fold_extra_rows"]:,} duplicate rows across {metrics["dup"]["fold_extra_groups"]:,} '
  f'groups that a case-sensitive title key misses ("Air Freight officer" / '
  f'"Air freight officer").')
add('')
add('---')
add('')

# ---- 6. enrichment -----------------------------------------------------------
enrich = metrics['enrichment']
add('## 6. Feature enrichment - alignment with `src/pipeline/`')
add('')
add(f'Implemented in **`{enrich["module"]}`** and imported by the notebook, so the pipeline and any other '
  f'consumer can use the same logic rather than reimplementing it.')
add('')
add(f'`feature_enrichment(df, job_category=None)` renames **{enrich["renamed"]}** source columns to the '
  f'names used by `src/pipeline/` and derives the feature columns `feature_engineer.py` adds, so '
  f'all **{enrich["schema_cols"]}** `JOBS_SCHEMA` columns it produces are satisfied and the cleaned '
  f'data can feed the existing dashboard without a translation layer. A further '
  f'**{len(enrich["dead_cols"])}** schema columns are dead: they are excluded from the module\'s '
  f'column list *and* never built (below). `schema_report()` from the same module backs the '
  f'assertions that fail the notebook if a live column is missing or a dead one reappears.')
add('')
add('| derived column | source | matches pipeline? |')
add('| --- | --- | --- |')
add('| `experience_level` | banded from `seniority_years` | yes - same thresholds |')
add('| `salary_band` | banded from `salary_max` | yes - same thresholds |')
add('| `skills`, `skill_count` | regex over `title` | yes - same vocabulary and boundary rule |')
add('| `sub_sector` | 2nd category from the bridge table | **no** - pipeline leaves it NULL |')
add('| `competitiveness_score` | p99 salary denominator | **no** - pipeline uses `max()` |')
add('')
add('`skill_count` and `competitiveness_score` are built here and then dropped in section 7 - they '
  'are listed because the enrichment produces them, not because they reach the file.')
add('| `job_id` | `metadata_jobPostId` | **no** - pipeline mints a fresh UUID |')
add('')
add(f'### {len(enrich["dead_cols"])} `JOBS_SCHEMA` columns deliberately not materialised')
add('')
add('These are absent from `JOBS_SCHEMA_COLUMNS` in `feature_enrichment.py`, so no caller can read '
  'the list and rebuild them; the full table definition still lives in `src/database/schema.py`.')
add('')
add('| column | value it would hold | why it is dead |')
add('| --- | --- | --- |')
for c, why in enrich['dead_cols'].items():
    add(f'| `{c}` | {DEAD_COL_VALUES[c]} | {why} |')
add('')
add('**Justification.** This is section 1.5\'s rule applied to the output side. Pruning '
  '`salary_type` for being constant and then inventing `salary_currency` two steps later would be '
  'incoherent: both encode the single fact *all salaries are monthly SGD*, which belongs in this '
  'report rather than in a million rows. `description` and `requirements` are worse than constant '
  '- the source CSV has no such fields, so they would be empty strings that read as "no '
  'requirements listed" rather than "never collected". `created_at` records when this notebook '
  'ran, not anything about the posting, and it would change the parquet\'s bytes on every '
  'execution with no change in the data.')
add('')
add('**Nothing downstream breaks.** `DatabaseManager.insert_jobs` back-fills `created_at` and '
  '`salary_currency` itself and filters `columns_order` to the columns actually present, printing '
  'anything absent; `location`, `description` and `requirements` are nullable in `JOBS_SCHEMA`. '
  'Both outputs are asserted free of these columns, and of any all-blank column, before they are '
  'written.')
add('')
add(f'**Two deliberate deviations**, both documented rather than silent:')
add('')
add(f'1. **`job_id` carries the real `metadata_jobPostId`.** The pipeline regenerates it as a UUID '
  f'at load time, which makes the processed rows impossible to join back to the raw layer or to '
  f'MCF - the audit trail the raw layer exists to provide becomes unreachable. Keeping the source '
  f'ID also lets `jobs_enriched.parquet` join to `job_category.parquet`.')
add(f'2. **`competitiveness_score` divides by the 99th percentile, not `max()`.** With the outlier '
  f'left in place (by design - see section 3.1), a `max()` denominator is dominated by it and '
  f'compresses the median row\'s salary contribution to roughly 0.02 of the 50 points available. '
  f'The p99 denominator gives a median score of **{enrich["comp_median"]}** - a better-scaled version '
  f'of a measure that section 7 then drops from the output entirely, for a reason no choice of '
  f'denominator fixes.')
add('')
add('### Two `feature_engineer.py` columns deliberately not reproduced')
add('')
add('**`days_posted`** is `now() - posting_date`. That measures when the ingestion process happened, '
  'not any property of the job posting, and it would make the saved parquet change on every '
  'execution - the file\'s bytes shifting with no change in the data. On a 2023-24 extract it '
  'resolves to the same value for every row sharing a posting date, so it carries no information '
  'beyond `posting_date`, which is already a column. `listing_days` '
  '(`expiry_date - posting_date`) is retained instead: deterministic, and the duration an analysis '
  'would actually ask for. Anything needing days-since-posting can compute it at query time '
  'against its own reference date.')
add('')
add('**`is_growth_role`** uses `count > median * 0.2`, which marks essentially every role in the '
  'file.')
add('')
add('Neither is part of `JOBS_SCHEMA`. Worth noting more broadly: of everything '
  '`feature_engineer.py` computes, **only `seniority_years` actually reaches the `jobs` table** - '
  '`salary_midpoint`, `salary_band`, `skill_count`, `competitiveness_score`, `days_posted` and '
  '`is_growth_role` are all dropped by `columns_order` at load time. `salary_midpoint` and '
  '`salary_band` are kept in `jobs_enriched.parquet` anyway, because it is meant to be usable '
  'directly for analysis rather than only as a database feed; `skill_count` and '
  '`competitiveness_score` are not, for the separate reason given in section 7.')
add('')
add(f'Coverage of the derived columns: `sub_sector` populated on '
  f'**{enrich["sub_sector_filled"]:,}** rows ({enrich["sub_sector_filled"]/metrics["final"]["rows"]*100:.1f}%), '
  f'`skills` matched on **{enrich["skills_found"]:,}** rows '
  f'({enrich["skills_found"]/metrics["final"]["rows"]*100:.1f}%, mean `skill_count` '
  f'{enrich["skill_count_mean"]}). The skills rate is low because **the source CSV has no description '
  f'column** - which is also why `description` is a dead column above - so the skill regex only '
  f'ever sees job titles. That limitation is inherited from the source data, but it means '
  f'`skill_count` and anything derived from it should be read as a title-keyword signal rather '
  f'than a requirements analysis.')
add('')
add(f'`jobs_enriched.parquet` carries {len(enrich["extra_cols"])} provenance columns beyond the `jobs` '
  f'table (`{"`, `".join(enrich["extra_cols"][:6])}`, ...) so a loader can `SELECT` just the '
  f'`JOBS_SCHEMA` columns while the cleaning decisions stay inspectable - and none of the '
  f'{len(enrich["dead_cols"])} dead ones.')
add('')
add('---')
add('')

# ---- 7. pruning derived columns ----------------------------------------------
prune = metrics['prune']
add('## 7. Pruning low-value derived columns')
add('')
add(f'Section 1.5 pruned columns that were dead on arrival. The same test is applied to the columns '
  f'this pipeline *creates*, and **{len(prune["columns"])}** of them fail it, costing '
  f'**{prune["mb"]} MB**:')
add('')
add(metrics['md_pruned'])
add('')
add(f'**`skill_count`** is a keyword match over job *titles*. `description` and `requirements` do '
  f'not exist in the source - they are dead columns (section 6) - so the regex only ever sees a '
  f'title, and a title is not where a posting lists its skills. The result is **{prune["skill_zero_pct"]}% '
  f'zeros** across **{prune["skill_distinct"]} distinct values**. A column that is almost entirely '
  f'one value cannot separate anything, and the non-zero remainder measures whether a recruiter '
  f'put "Python" in the headline, not whether the job needs Python.')
add('')
add(f'**`competitiveness_score`** is meant to rank how contested a posting is. Correlated against '
  f'the quantities that actually express contest, it returns '
  f'**{prune["comp_corr"]["applications"]:+.2f}** with applications, '
  f'**{prune["comp_corr"]["views"]:+.2f}** with views and '
  f'**{prune["comp_corr"]["vacancies"]:+.2f}** with vacancies. A score that does not track its own '
  f'subject is worse than absent, because a dashboard presents it as though it does. Note this is '
  f'a verdict on the *measure*, not on the p99 denominator argued for in section 6.2 - that fix '
  f'improves the scaling of a quantity that still fails to correlate with anything it claims to '
  f'summarise.')
add('')
add('Both are **computed and then dropped** rather than removed from `feature_enrichment.py`: that '
  'module is shared production code, and `competitiveness_score` needs `skill_count` to build. '
  'What this step decides is only what is worth *storing*.')
add('')
add(f'**`skills` is kept**, despite {prune["skills_not_specified_pct"]}% of rows reading '
  f'"Not Specified" across {prune["skills_distinct"]} distinct values and {prune["skills_mb"]} MB - it '
  f'is a `JOBS_SCHEMA` column, so dropping it would break the loader contract. It carries the '
  f'same title-only limitation as `skill_count` and should be read the same way: a '
  f'headline-keyword signal, not a requirements analysis. Worth revisiting when the schema next '
  f'changes.')
add('')
add('---')
add('')

# ---- appendix ----------------------------------------------------------------
add('## Appendix - final column profile')
add('')
add(metrics['md_profile_final'])
add('')
add('### Outputs')
add('')
for f, n in metrics['outputs'].items():
    add(f'- `data/processed/{f}` - {n:,} rows')

REPORT_PATH.write_text('\n'.join(lines))
print(f'wrote {REPORT_PATH}  ({len(lines)} lines, {REPORT_PATH.stat().st_size/1024:.1f} KB)')

wrote ../docs/data-cleaning-report-generated.md  (309 lines, 35.3 KB)


---

**Next step:** the analysis layer reads `data/processed/jobs_enriched.parquet` (pipeline column
names, joinable to `job_category.parquet` on `job_id`) or `jobs_cleaned.parquet` for the source
names. Salary aggregates must filter `salary_max.notna()` **and** `salary_flag != 'outlier'` and
quote the coverage rate; employer rankings should deduplicate with `drop_duplicates('dup_group_id')`
at query time. The case-folded title is not stored — write `title.str.lower()` if an aggregation
needs it.